In [ ]:
# The overall workflow: TRGT spanning BAM → select C9orf72 reads
# → match read IDs to original HiFi BAM → use FL to trim flanks 
# → locate the exact repeat region in the original read → extract corresponding Phred scores 
# → calculate A+T% → calculate mean repeat-region quality → Pearson/Spearman correlation. 
# Keep Hifi_read, The correlation with A+T average and reads quality 

# =============================================================================
# STEP 1 - Import python packages:
# =============================================================================

from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysam
from IPython.display import display
from scipy.stats import pearsonr, spearmanr

# =============================================================================
# STEP 2 — Define input files, sample, and target repeat locus
# =============================================================================
#
# spanning_bam:
#     TRGT spanning-read BAM. This identifies reads spanning the target
#     tandem-repeat locus and provides TRGT annotations such as TR and FL. FL here is defined as [-50,50] by ptcp pipeline
#
# hifi_bam:
#     Original PacBio HiFi alignment BAM. This is used to recover the
#     original sequence and per-base Phred quality scores.
#
# target_locus:
#     Only analyze reads assigned by TRGT to FTDALS1_C9orf72.
#
# min_repeat_length:
#     Require at least one base after removing the TRGT-defined flanks.
# =============================================================================


# Run from the notebook directory, or set data_dir to its absolute path.
data_dir = Path.cwd()
spanning_bam = data_dir / '*.trgt.sorted.spanning.bam'  # The trgt.sorted.spanning.bam from ptcp pipeline
hifi_bam = data_dir / '*.hifi.pbmm2.bam' # The pbmm2 aligned bam file 
sample_name = '*'
target_locus = 'FTDALS1_C9orf72' # The target regions
min_repeat_length = 1 # In case some reads have flanking regions, but do not have repeats
output_prefix = data_dir / '*_C9orf72_AT_content_vs_core_quality'
for path in (spanning_bam, hifi_bam):
    if not path.is_file():
        raise FileNotFoundError(f'Set data_dir to the notebook directory: {path}')


# =============================================================================
# STEP 3 — Define sequence-orientation helper functions
# =============================================================================
#
# PacBio/TRGT sequences can appear in either orientation.
#
# revcomp():
#     Generates the reverse-complement sequence.
#
# unique_span_match():
#     Searches for the complete TRGT spanning sequence inside the original
#     HiFi read in BOTH forward and reverse-complement orientations.
#
#     Three possible outcomes:
#
#       success
#           Exactly one sequence match is found.
#
#       sequence_not_found
#           The TRGT spanning sequence cannot be recovered from the
#           corresponding original HiFi sequence.
#
#       ambiguous_sequence_match
#           More than one possible sequence match exists, so the repeat
#           coordinates cannot be assigned uniquely.
#
# Reads without a unique match are excluded rather than making an uncertain
# coordinate assignment.
# =============================================================================


complement = str.maketrans('ACGTRYMKBDHVN', 'TGCAYRKMVHDBN')
def revcomp(seq):
    return seq.translate(complement)[::-1]

def unique_span_match(full_seq, span_seq):
    hits = []
    for orientation, needle in [('forward', span_seq), ('reverse', revcomp(span_seq))]:
        start = full_seq.find(needle)
        while start != -1:
            hits.append((start, orientation))
            if len(hits) > 1:
                return None, 'ambiguous_sequence_match'
            start = full_seq.find(needle, start + 1)
    return (hits[0], 'success') if hits else (None, 'sequence_not_found')


# =============================================================================
# STEP 4 — Define sequence-orientation helper functions
# =============================================================================
#
# PacBio/TRGT sequences can appear in either orientation.
#
# revcomp():
#     Generates the reverse-complement sequence.
#
# unique_span_match():
#     Searches for the complete TRGT spanning sequence inside the original
#     HiFi read in BOTH forward and reverse-complement orientations.
#
#     Three possible outcomes:
#
#       success
#           Exactly one sequence match is found.
#
#       sequence_not_found
#           The TRGT spanning sequence cannot be recovered from the
#           corresponding original HiFi sequence.
#
#       ambiguous_sequence_match
#           More than one possible sequence match exists, so the repeat
#           coordinates cannot be assigned uniquely.
#
# Reads without a unique match are excluded rather than making an uncertain
# coordinate assignment.
# =============================================================================


spanning_reads = {}
with pysam.AlignmentFile(spanning_bam, 'rb', check_sq=False) as bam:
    for read in bam.fetch(until_eof=True):
        if not read.has_tag('TR') or read.get_tag('TR') != target_locus:
            continue
        if read.is_secondary or read.is_supplementary:
            continue
        if read.query_name in spanning_reads:
            raise ValueError(f'Duplicate target spanning read: {read.query_name}')
        spanning_reads[read.query_name] = read
if not spanning_reads:
    raise ValueError(f'No target spanning reads found for {target_locus}')
print(f'{sample_name}: {len(spanning_reads):,} unique {target_locus} spanning reads')


# =============================================================================
# STEP 5 — Match each TRGT spanning read to its original PacBio HiFi read
# =============================================================================
#
# Scan the original HiFi BAM and retain reads whose read IDs are present in
# the C9orf72 TRGT spanning-read set.
#
# IMPORTANT:
# This uses READ-ID matching rather than genomic coordinates.
#
# Therefore, this step does not depend on assumptions about the reference
# genome coordinates used to define the repeat locus.
#
# Only primary HiFi alignments are used.
# =============================================================================

rows, qc_rows = [], []
matched_ids = set()
# Full sequential scan: no genome-build/coordinate assumption; index not required.
with pysam.AlignmentFile(hifi_bam, 'rb', check_sq=False) as bam:
    for original in bam.fetch(until_eof=True):
        read_id = original.query_name
        if read_id not in spanning_reads or original.is_secondary or original.is_supplementary:
            continue
        if read_id in matched_ids:
            raise ValueError(f'Duplicate primary HiFi read: {read_id}')
        matched_ids.add(read_id)
        span = spanning_reads[read_id]
        qc = {'read_id': read_id, 'status': 'success'}
        qc_rows.append(qc)


        # =====================================================================
        # STEP 6 — Basic sequence and base-quality QC
        # =====================================================================
        #
        # Both TRGT and original HiFi sequences must be available.
        #
        # The original HiFi read must also contain one valid Phred quality
        # score for every base.
        #
        # Reads containing unavailable/invalid quality values are removed
        # because repeat-region mean quality cannot be calculated reliably.
        # =====================================================================

        if not span.query_sequence or not original.query_sequence:
            qc['status'] = 'missing_sequence'
            continue
        span_seq = span.query_sequence.upper()
        full_seq = original.query_sequence.upper()
        qualities = original.query_qualities
        if qualities is None or len(qualities) != len(full_seq) or 255 in qualities:
            qc['status'] = 'missing_or_invalid_hifi_qualities'
            continue


        # =====================================================================
        # STEP 7 — Use the TRGT FL tag to define the repeat/core sequence
        # =====================================================================
        #
        # TRGT spanning reads contain flanking sequence surrounding the repeat.
        #
        # The FL tag records the number of bases belonging to the left and
        # right flanks:
        #
        #       |------ spanning sequence ------|
        #
        #       [left flank][ repeat ][right flank]
        #        <---FL--->           <---FL--->
        #
        # Removing these flanks gives the "core" sequence used in this
        # analysis.
        #
        # IMPORTANT:
        # We use the FL values stored for each read instead of assuming that
        # every read has a fixed 50-bp flank.
        # =====================================================================
            
        # FL belongs to the stored spanning sequence.
        try:
            flanks = span.get_tag('FL')
            if len(flanks) != 2:
                raise ValueError
            left, right = map(int, flanks)
            if min(left, right) < 0:
                raise ValueError
        except (KeyError, TypeError, ValueError):
            qc['status'] = 'missing_or_invalid_FL'
            continue
        end = len(span_seq) - right
        core_seq = span_seq[left:end] if end > left else ''
        if len(core_seq) < min_repeat_length:
            qc['status'] = 'core_too_short'
            continue

        # =====================================================================
        # STEP 8 — Locate the TRGT spanning sequence in the original HiFi read
        # =====================================================================
        #
        # Search for the complete spanning sequence in the original HiFi read.
        #
        # The match can occur in either:
        #
        #       forward orientation
        #
        # or
        #
        #       reverse-complement orientation
        #
        # Once the spanning sequence is located, the FL information is used
        # to convert the repeat/core boundaries into coordinates on the
        # original HiFi read.
        #
        # This is important because the base-quality scores must be extracted
        # from exactly the same bases used for the A+T calculation.
        # =====================================================================


        match, status = unique_span_match(full_seq, span_seq)
        if match is None:
            qc['status'] = status
            continue
        start, orientation = match
        if orientation == 'forward':
            core_start, core_end = start + left, start + end
        else:
            core_start, core_end = start + right, start + len(span_seq) - left


        # =====================================================================
        # STEP 9 — Verify the recovered repeat sequence and extract its quality
        # =====================================================================
        #
        # Recover the sequence from the calculated HiFi coordinates and verify
        # that it exactly matches the TRGT-defined core sequence.
        #
        # For reverse-orientation reads, compare against the reverse complement.
        #
        # This assertion acts as an internal sanity check that the sequence and
        # quality coordinates are aligned correctly.
        #
        # The corresponding per-base Phred scores are then extracted from the
        # original HiFi read.
        # =====================================================================

        
        recovered_core = full_seq[core_start:core_end]
        assert recovered_core == (core_seq if orientation == 'forward' else revcomp(core_seq))
        core_quality = np.asarray(qualities[core_start:core_end], dtype=float)
        assert len(core_quality) == len(core_seq)


        # =====================================================================
        # STEP 10 — Calculate repeat-region nucleotide composition
        # =====================================================================
        #
        # Count A, C, G, T, and N within the trimmed repeat/core sequence.
        #
        # A+T percentage is calculated as:
        #
        #                A + T
        #       ------------------------- × 100
        #          A + T + C + G
        #
        # Noncanonical/ambiguous bases are therefore NOT included in the
        # denominator.
        #
        # This provides a per-read measurement of how A/T-rich the repeat
        # sequence is.
        # =====================================================================

        
        counts = Counter(core_seq)
        canonical = sum(counts[b] for b in 'ACGT')
        if canonical == 0:
            qc['status'] = 'no_canonical_bases'
            continue


        # =====================================================================
        # STEP 11 — Store per-read repeat and quality measurements
        # =====================================================================
        #
        # For every successfully processed read, record:
        #
        #   • read ID
        #   • sequence orientation
        #   • full HiFi read length
        #   • TRGT spanning-read length
        #   • left/right flank lengths
        #   • trimmed repeat/core length
        #   • core coordinates within the original HiFi read
        #   • A/C/G/T/N counts
        #   • number of ambiguous bases
        #   • A+T percentage
        #   • mean Phred quality within the repeat/core
        #   • mean Phred quality across the entire HiFi read
        #
        # mean_core_quality is the main quality variable used in the
        # subsequent correlation analysis.
        # =====================================================================

        
        
        rows.append({
            'read_id': read_id, 'orientation': orientation,
            'hifi_read_length': len(full_seq), 'spanning_length': len(span_seq),
            'flank_left': left, 'flank_right': right, 'core_length': len(core_seq),
            'hifi_core_start_0based': core_start, 'hifi_core_end_exclusive': core_end,
            **{f'{b.lower()}_count': counts[b] for b in 'ACGTN'},
            'ambiguous_base_count': len(core_seq) - canonical,
            'at_percentage': 100 * (counts['A'] + counts['T']) / canonical,
            # Arithmetic mean of BAM base Phred scores, not MAPQ or rq.
            'mean_core_quality': float(core_quality.mean()),
            'mean_hifi_read_quality': float(np.mean(qualities)),
        })


# =============================================================================
# STEP 12 — Generate QC summary and save excluded-read information
# =============================================================================
#
# Any TRGT spanning read that was never found as a primary read in the
# original HiFi BAM is labeled "not_in_hifi_primary".
#
# The QC table allows us to see exactly how many reads were retained or
# removed at each step instead of silently dropping problematic reads.
# =============================================================================

qc_rows.extend({'read_id': rid, 'status': 'not_in_hifi_primary'}
               for rid in sorted(set(spanning_reads) - matched_ids))
df_quality = pd.DataFrame(rows)
df_quality_qc = pd.DataFrame(qc_rows).sort_values('read_id')
df_quality_qc.to_csv(f'{output_prefix}_QC.csv', index=False)
print(f'Exact read-ID overlap with primary HiFi reads: {len(matched_ids):,}')
print(df_quality_qc['status'].value_counts().to_string())
if df_quality.empty:
    raise ValueError('No usable repeat measurements; inspect the QC CSV.')


# =============================================================================
# STEP 13 — Save and inspect the successfully processed per-read dataset
# =============================================================================
#
# The final per-read CSV contains only reads that passed all QC criteria.
#
# The descriptive statistics provide a quick check of:
#
#   • repeat/core length distribution
#   • A+T percentage distribution
#   • repeat-region mean Phred quality distribution
# =============================================================================


df_quality = df_quality.sort_values('read_id').reset_index(drop=True)
df_quality.to_csv(f'{output_prefix}_per_read.csv', index=False)
display(df_quality.head(10))
display(df_quality[['core_length', 'at_percentage', 'mean_core_quality']].describe())


# =============================================================================
# STEP 14 — Test the relationship between A+T content and base quality
# =============================================================================
#
# X-axis:
#     A+T percentage within the trimmed repeat/core sequence.
#
# Y-axis:
#     Mean per-base Phred quality within exactly the same repeat/core region.
#
# Two complementary correlation statistics are calculated:
#
# Pearson r
#     Tests the strength of a LINEAR relationship between A+T content
#     and mean quality.
#
# Spearman rho
#     Tests whether there is a MONOTONIC relationship and is less dependent
#     on linearity and the exact distribution of the measurements.
#
# A linear regression line is included as a visual representation of the
# overall trend.
# =============================================================================


x = df_quality['at_percentage'].to_numpy()
y = df_quality['mean_core_quality'].to_numpy()
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.scatter(x, y, s=18, alpha=0.35, color='#2b5c8f', edgecolors='none',
           rasterized=True, label=f'Matched reads (n={len(x):,})')

# =============================================================================
# STEP 15 — Calculate correlation statistics and linear trend
# =============================================================================
#
# Correlations are calculated only when:
#
#   • at least 3 reads are available;
#   • A+T percentage is not constant;
#   • mean quality is not constant.
#
# This avoids calculating undefined or meaningless correlations.
# =============================================================================

if len(x) >= 3 and np.ptp(x) > 0 and np.ptp(y) > 0:
    pearson_r, pearson_p = pearsonr(x, y)
    spearman_rho, spearman_p = spearmanr(x, y)
    line_x = np.linspace(x.min(), x.max(), 200)
    ax.plot(line_x, np.polyval(np.polyfit(x, y, 1), line_x),
            color='#e27b38', lw=2, label='Linear trend')
    statistics_text = (f'Pearson r = {pearson_r:.3f}, p = {pearson_p:.2e}\n'
                       f'Spearman ρ = {spearman_rho:.3f}, p = {spearman_p:.2e}')
else:
    statistics_text = 'Correlation unavailable: too few reads or constant values'
print(statistics_text)

# =============================================================================
# STEP 16 — Annotate, display, and save the final figure
# =============================================================================
#
# Each point represents ONE C9orf72 spanning read.
#
# Therefore, the final figure asks:
#
#     "Across individual C9orf72-spanning PacBio HiFi reads,
#      is increasing A+T content in the trimmed repeat region
#      associated with a change in the base-call quality of
#      that same region?"
#
# The PNG version is saved at 300 dpi for figures/presentations.
# The SVG version is retained as a vector graphic for later editing.
# =============================================================================


ax.text(0.04, 0.96, statistics_text, transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
ax.set(xlabel='A+T content in Trimmed Repeat Region',
       ylabel='Mean Phred Qualty of Trimmed Repeat Region',
       title=f'{sample_name} {target_locus}: A+T Content vs Mean Quality A+T divided by A+T+C+G')
ax.grid(alpha=0.25, linestyle='--')
ax.set_axisbelow(True)
ax.legend(loc='best')
fig.tight_layout()
fig.savefig(f'{output_prefix}.png', dpi=300, bbox_inches='tight')
fig.savefig(f'{output_prefix}.svg', bbox_inches='tight')
plt.show()


In [ ]:
# Layer1: TRGT spanning read → FL trimming → repeat/core sequence → gene orientation → repeat length + A/T% + G4C2 motif count
# Layer2: read ID → original HiFi BAM → exact spanning-sequence match → locate the same repeat region → extract per-base Phred scores → mean repeat-region quality


# =============================================================================
# C9orf72 — Raw Per-Read Repeat Metrics
# =============================================================================
#
# PURPOSE
# -------
# Generate a per-read dataset for all valid C9orf72 TRGT spanning reads.
#
# For each spanning read, this pipeline:
#
#   1. Identifies the C9orf72 TRGT spanning record.
#   2. Uses the TRGT FL tag to remove left/right flanking sequence.
#   3. Orients the sequence into the target gene direction.
#   4. Calculates repeat length.
#   5. Calculates A+T percentage.
#   6. Counts G4C2-related repeat motifs.
#   7. Matches the spanning read back to the original HiFi BAM.
#   8. When possible, recovers the per-base HiFi quality scores for exactly
#      the same repeat/core region.

# =============================================================================
# STEP 1 — Import required packages
# =============================================================================
#
# pathlib:
#     File/path handling.
#
# Counter:
#     Count individual A/C/G/T bases.
#
# re:
#     Count occurrences of GGGGCC/GGCCCC motifs.
#
# matplotlib:
#     Used later for visualization.
#
# numpy / pandas:
#     Numerical calculations and tabular data handling.
#
# pysam:
#     Read BAM files and access TRGT/PacBio tags and base qualities.
#
# scipy.stats:
#     Used later for Pearson/Spearman correlation analyses.
# =============================================================================
from pathlib import Path
from collections import Counter
import re

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import pysam
from IPython.display import display
from scipy.stats import pearsonr, spearmanr


# =============================================================================
# STEP 2 — Define sample, input BAM files, and target locus
# =============================================================================
#
# spanning_bam:
#     TRGT spanning-read BAM containing reads assigned to tandem-repeat loci.
#
# hifi_bam:
#     Original PacBio HiFi alignment BAM. Used to recover the original
#     sequence and per-base Phred quality scores.
#
# target_locus:
#     Analyze only reads assigned by TRGT to FTDALS1_C9orf72.
#
# target_strand:
#     C9orf72 is treated using the "-" gene-direction convention used in
#     this notebook.
#
# max_display_reads:
#     Controls later track visualization. None means display all retained
#     reads.
# =============================================================================


data_dir = Path.cwd()

sample_name = "*"
spanning_bam = data_dir / (
    "*.trgt.sorted.spanning.bam"
)
hifi_bam = data_dir / "*.hifi.pbmm2.bam"

target_locus = "FTDALS1_C9orf72"
target_strand = "-"

# None displays every retained read in the track figures.
max_display_reads = None


# =============================================================================
# STEP 3 — Define sequence orientation helper functions
# =============================================================================
#
# revcomp():
#     Generate the reverse complement of a DNA sequence.
#
# locate_spanning_sequence():
#     Search for the complete TRGT spanning sequence within the corresponding
#     original HiFi read.
#
#     Because BAM sequences may appear in different orientations, both the
#     spanning sequence and its reverse complement are searched.
#
# Possible outcomes:
#
#     "success"
#         Exactly one match was identified.
#
#     "sequence_not_found"
#         No exact match was found in the original HiFi sequence.
#
#     "ambiguous_sequence_match"
#         More than one possible match was found. The location therefore
#         cannot be assigned uniquely.
#
# Only a unique match is used for subsequent HiFi quality recovery.
# =============================================================================


complement = str.maketrans("ACGTRYMKBDHVN", "TGCAYRKMVHDBN")

def revcomp(sequence):
    return sequence.translate(complement)[::-1]


def locate_spanning_sequence(full_sequence, spanning_sequence):
    """ Find a unique full spanning-sequence match in either orientation."""
    hits = []

    for orientation, needle in [
        ("forward", spanning_sequence),
        ("reverse", revcomp(spanning_sequence)),
    ]:
        start = full_sequence.find(needle)

        while start >= 0:
            hits.append((start, orientation))

            if len(hits) > 1:
                return None, "ambiguous_sequence_match"

            start = full_sequence.find(needle, start + 1)

    if not hits:
        return None, "sequence_not_found"

    return hits[0], "success"


# =============================================================================
# STEP 4 — Confirm that the required BAM files exist
# =============================================================================
#
# Stop immediately if either the TRGT spanning BAM or original HiFi BAM
# cannot be found.
#
# This prevents the downstream analysis from running with an incorrect or
# missing input file.
# =============================================================================


for path in (spanning_bam, hifi_bam):
    if not path.is_file():
        raise FileNotFoundError(path)

# =============================================================================
# STEP 5 — Collect unique primary C9orf72 TRGT spanning reads
# =============================================================================
#
# Scan the TRGT spanning BAM and select reads satisfying ALL of the following:
#
#   1. The read contains a TR tag.
#   2. TR == FTDALS1_C9orf72.
#   3. The record is not secondary.
#   4. The record is not supplementary.
#
# Each read is stored using its PacBio read ID.
#
# The analysis assumes one primary C9orf72 spanning record per read ID.
# Therefore, duplicate primary records are treated as an error rather than
# silently choosing one.
# =============================================================================


# Collect unique primary C9orf72 spanning records.
spanning_reads = {}

with pysam.AlignmentFile(spanning_bam, "rb", check_sq=False) as bam:
    for read in bam.fetch(until_eof=True):
        if not read.has_tag("TR"):
            continue
        if read.get_tag("TR") != target_locus:
            continue
        if read.is_secondary or read.is_supplementary:
            continue

        if read.query_name in spanning_reads:
            raise ValueError(
                f"Duplicate primary spanning record: {read.query_name}"
            )

        spanning_reads[read.query_name] = read

if not spanning_reads:
    raise ValueError("No C9orf72 spanning records found.")


# =============================================================================
# STEP 6 — Identify which TRGT spanning reads are present in the HiFi BAM
# =============================================================================
#
# Match reads between the TRGT spanning BAM and original HiFi BAM using
# EXACT READ IDs.
#
# IMPORTANT:
# No sequencing-quality threshold is applied here.
#
# overlap_ids:
#     Contains every C9orf72 spanning read ID found anywhere in the HiFi BAM.
#
# original_reads:
#     Contains one PRIMARY HiFi record for each matching read. These primary
#     records will later be used to recover the original sequence and
#     per-base quality values.
#
# Secondary/supplementary HiFi records can establish read-ID membership in
# overlap_ids, but they are not used for quality extraction.
# =============================================================================

# HiFi membership uses exact read IDs, without a quality threshold.
overlap_ids = set()
original_reads = {}

with pysam.AlignmentFile(hifi_bam, "rb", check_sq=False) as bam:
    for read in bam.fetch(until_eof=True):
        if read.query_name not in spanning_reads:
            continue

        overlap_ids.add(read.query_name)

        # Use one primary record for recovering quality/modification data.
        if read.is_secondary or read.is_supplementary:
            continue

        if read.query_name in original_reads:
            raise ValueError(
                f"Duplicate primary HiFi record: {read.query_name}"
            )

        original_reads[read.query_name] = read

# =============================================================================
# STEP 7 — Initialize per-read measurement and QC tables
# =============================================================================
#
# records:
#     Stores the final biological/technical measurements for retained
#     C9orf72 repeat cores.
#
# qc_records:
#     Stores QC information explaining whether each spanning read was
#     retained and whether its HiFi quality could be recovered.
#
# Every C9orf72 spanning read is evaluated individually.
# =============================================================================


records = []
qc_records = []

for read_id in sorted(spanning_reads):
    qc = {"read_id": read_id}
    qc_records.append(qc)

# =========================================================================
# STEP 8 — Require the spanning read ID to occur in the HiFi BAM
# =========================================================================
#
# Reads that cannot be found in the original HiFi BAM are recorded in QC
# and excluded from the retained repeat dataset.
#
# At this stage, this is only a READ-ID membership check; no quality
# filtering is performed.
# =========================================================================
    
    if read_id not in overlap_ids:
        qc["status"] = "not_in_hifi"
        continue

    span = spanning_reads[read_id]

    if not span.query_sequence:
        qc["status"] = "missing_spanning_sequence"
        continue

    span_sequence = span.query_sequence.upper()

# =========================================================================
# STEP 9 — Read the TRGT FL tag
# =========================================================================
#
# The TRGT spanning sequence contains:
#
#        left flank | repeat/core | right flank
#
# The FL tag specifies the lengths of the left and right flanking
# sequences for this particular spanning record.
#
# These values are used directly instead of assuming a fixed flank size.
#
# The FL tag must:
#
#   • exist;
#   • contain exactly two values;
#   • contain non-negative flank lengths.
# =========================================================================
    
    try:
        flanks = span.get_tag("FL")
        if len(flanks) != 2:
            raise ValueError

        left, right = map(int, flanks)

        if min(left, right) < 0:
            raise ValueError

    except (KeyError, TypeError, ValueError):
        qc["status"] = "missing_or_invalid_FL"
        continue

# =========================================================================
# STEP 10 — Remove TRGT flanks and define the repeat/core sequence
# =========================================================================
#
# The core is obtained by removing:
#
#     left bases from the beginning
#     right bases from the end
#
# Therefore:
#
#     span_sequence
#     |-------------------------------------|
#     | left flank |   repeat   | right flank |
#                    ↑
#               reference_core
#
# A core is retained as long as it contains at least ONE base.
#
# This is intentionally permissive: short cores are NOT removed here.
# =========================================================================
    
    core_end = len(span_sequence) - right

    if core_end <= left:
        qc["status"] = "empty_or_invalid_core"
        continue

    # Keep ALL nonempty cores, including 1 bp.
    reference_core = span_sequence[left:core_end]


# =========================================================================
# STEP 11 — Orient the sequence into the target gene direction
# =========================================================================
#
# The stored BAM sequence orientation is not necessarily the most
# intuitive orientation for biological visualization.
#
# For the current C9orf72 analysis:
#
#     target_strand = "-"
#
# Therefore, the spanning sequence and repeat/core sequence are reverse
# complemented so that downstream sequence tracks follow the same
# gene-direction convention used elsewhere in the notebook.
#
# When the sequence is reversed, the left and right flanks exchange
# positions.
#
# IMPORTANT:
# This orientation operation does NOT change repeat length or A+T content.
# It standardizes how sequences are represented and plotted.
# =========================================================================

    
    # Same gene-direction convention as the existing notebook.
    if target_strand == "-":
        gene_sequence = revcomp(span_sequence)
        gene_left, gene_right = right, left
        core_sequence = revcomp(reference_core)
    else:
        gene_sequence = span_sequence
        gene_left, gene_right = left, right
        core_sequence = reference_core

# =========================================================================
# STEP 12 — Calculate nucleotide composition and A+T percentage
# =========================================================================
#
# Count A, C, G, and T within the gene-oriented repeat/core sequence.
#
# canonical_count:
#
#       A + C + G + T
#
# A+T percentage:
#
#               A + T
#       --------------------- × 100
#         A + T + C + G
#
# Ambiguous/noncanonical bases are therefore excluded from the
# denominator.
#
# If the core contains no canonical A/C/G/T bases, A+T percentage is
# recorded as NaN rather than forcing a numerical value.
# =========================================================================

  
    counts = Counter(core_sequence)
    canonical_count = sum(counts[b] for b in "ACGT")

    at_percentage = (
        100 * (counts["A"] + counts["T"]) / canonical_count
        if canonical_count else np.nan
    )

# =========================================================================
# STEP 13 — Count canonical C9orf72 repeat motifs
# =========================================================================
#
# Count occurrences of the two relevant 6-bp sequence representations:
#
#     GGGGCC
#     GGCCCC
#
# Regex lookahead (?=...) is used so that motif detection allows
# overlapping matches.
#
# motif_count is defined as:
#
#     max(count_fwd, count_rev)
#
# This provides one motif-count measurement per read while accounting
# for the two sequence representations used in the analysis.
# =========================================================================


    count_fwd = len(re.findall(r"(?=GGGGCC)", core_sequence))
    count_rev = len(re.findall(r"(?=GGCCCC)", core_sequence))

    # =========================================================================
    # STEP 14 — Create the basic per-read repeat record
    # =========================================================================
    #
    # At this point, the repeat itself has been successfully characterized.
    #
    # Store:
    #
    #   • read ID
    #   • spanning-read length
    #   • repeat/core length
    #   • original FL values
    #   • gene-oriented repeat sequence
    #   • gene-oriented left/right flanking sequences
    #   • GGGGCC count
    #   • GGCCCC count
    #   • final motif count
    #   • A+T percentage
    #
    # HiFi quality-related fields are initially empty/NaN. They will be
    # populated only if the original HiFi sequence and qualities can be
    # recovered successfully.
    # =========================================================================

    row = {
        "read_id": read_id,
        "spanning_length": len(span_sequence),
        "repeat_length": len(core_sequence),
        "left_flank": left,
        "right_flank": right,
        "core_seq": core_sequence,
        "gene_left_flank": gene_sequence[:gene_left],
        "gene_right_flank": (
            gene_sequence[len(gene_sequence) - gene_right:]
            if gene_right else ""
        ),
        "count_fwd": count_fwd,
        "count_rev": count_rev,
        "motif_count": max(count_fwd, count_rev),
        "at_percentage": at_percentage,
        "mean_core_quality": np.nan,
        "core_qualities": None,
        "orientation": None,
        "hifi_core_start": None,
        "hifi_core_end": None,
    }

    # =========================================================================
    # STEP 15 — Mark the repeat/core itself as successfully retained
    # =========================================================================
    #
    # "retained" means that a valid, nonempty TRGT-defined repeat/core has
    # been obtained.
    #
    # This is intentionally separate from "quality_status".
    #
    # A read can therefore have:
    #
    #     status = retained
    #
    # while HiFi quality recovery fails later.
    #
    # This distinction prevents missing quality information from being
    # mistaken for an invalid repeat measurement.
    # =========================================================================
    
    qc["status"] = "retained"
    qc["quality_status"] = "no_primary_hifi_record"

    original = original_reads.get(read_id)

    # =========================================================================
    # STEP 16 — Recover the corresponding original HiFi sequence
    # =========================================================================
    #
    # If a primary HiFi record exists, attempt to locate the complete TRGT
    # spanning sequence within that original read.
    #
    # Exact sequence matching is performed in both forward and reverse
    # orientations.
    #
    # Successful matching allows us to determine exactly where the
    # TRGT-defined repeat/core occurs within the original HiFi read.
    # =========================================================================

    
    if original is not None:
        if not original.query_sequence:
            qc["quality_status"] = "missing_hifi_sequence"
        else:
            match, match_status = locate_spanning_sequence(
                original.query_sequence.upper(), span_sequence
            )
            qc["quality_status"] = match_status

            # =================================================================
            # STEP 17 — Convert the TRGT core boundaries to HiFi-read coordinates
            # =================================================================
            #
            # Once the spanning sequence is located, calculate the exact
            # repeat/core interval within the original HiFi read.
            #
            # Forward match:
            #
            #     core begins after the TRGT left flank.
            #
            # Reverse match:
            #
            #     left/right relationships are reversed because the spanning
            #     sequence matches the reverse-complement orientation.
            #
            # q_start and q_end therefore define the same biological core
            # region in the original HiFi sequence.
            # =================================================================
            
            if match is not None:
                start, orientation = match

                if orientation == "forward":
                    q_start = start + left
                    q_end = start + core_end
                else:
                    q_start = start + right
                    q_end = start + len(span_sequence) - left

                # =============================================================
                # STEP 18 — Sanity-check the recovered HiFi core sequence
                # =============================================================
                #
                # Verify that the sequence recovered from the calculated HiFi
                # coordinates exactly equals the expected TRGT core sequence.
                #
                # For a reverse match, the expected sequence is the reverse
                # complement of reference_core.
                #
                # This assertion ensures that subsequent quality values are
                # extracted from exactly the intended repeat bases.
                # =============================================================
                
                expected = (
                    reference_core
                    if orientation == "forward"
                    else revcomp(reference_core)
                )
                assert original.query_sequence.upper()[q_start:q_end] == expected

                row.update({
                    "orientation": orientation,
                    "hifi_core_start": q_start,
                    "hifi_core_end": q_end,
                })

                # =============================================================
                # STEP 19 — Extract per-base Phred quality from the HiFi core
                # =============================================================
                #
                # original.query_qualities contains one Phred quality score
                # for each base of the original HiFi read.
                #
                # Extract only:
                #
                #     qualities[q_start:q_end]
                #
                # so that the quality measurements correspond exactly to the
                # same TRGT-defined repeat/core used for sequence analysis.
                #
                # This is BASE quality, not:
                #
                #     • BAM MAPQ
                #     • PacBio rq
                #     • a whole-read quality threshold
                # =============================================================
                
                qualities = original.query_qualities

                if qualities is None or len(qualities) != len(
                    original.query_sequence
                ):
                    qc["quality_status"] = "missing_or_invalid_qualities"
                else:
                    core_q = np.asarray(
                        qualities[q_start:q_end], dtype=float
                    )

                    # =========================================================
                    # STEP 20 — Orient quality scores to match the sequence
                    # =========================================================
                    #
                    # Quality scores must undergo the same orientation changes
                    # as the sequence.
                    #
                    # First:
                    # If the spanning sequence matched the HiFi read in reverse
                    # orientation, reverse the quality vector so that its order
                    # corresponds to the stored spanning sequence.
                    #
                    # Second:
                    # Because C9orf72 is being displayed in the "-" gene
                    # direction, reverse the quality vector again to match
                    # core_sequence.
                    #
                    # After these transformations:
                    #
                    #     core_q[i]
                    #
                    # corresponds directly to:
                    #
                    #     core_sequence[i]
                    #
                    # for every base position.
                    # =========================================================

                    
                    # Map HiFi quality order to spanning sequence order.
                    if orientation == "reverse":
                        core_q = core_q[::-1]

                    # Then map to gene direction.
                    if target_strand == "-":
                        core_q = core_q[::-1]

                    assert len(core_q) == len(core_sequence)

                    # =========================================================
                    # STEP 21 — Remove unavailable quality measurements
                    # =========================================================
                    #
                    # A Phred value of 255 represents unavailable base quality.
                    #
                    # If any base within the core has unavailable quality, this
                    # read is flagged rather than calculating a potentially
                    # misleading mean quality.
                    #
                    # Otherwise:
                    #
                    #   core_qualities
                    #       retains every per-base Phred score.
                    #
                    #   mean_core_quality
                    #       is the arithmetic mean of those scores.
                    # =========================================================

                    # 255 denotes unavailable base quality.
                    if np.any(core_q == 255):
                        qc["quality_status"] = "unavailable_core_qualities"
                    else:
                        row["core_qualities"] = core_q
                        row["mean_core_quality"] = float(core_q.mean())
                        qc["quality_status"] = "success"

    # =========================================================================
    # STEP 22 — Add the retained read to the final raw-metric dataset
    # =========================================================================
    #
    # IMPORTANT:
    #
    # The row is retained even if HiFi quality recovery failed.
    #
    # Therefore:
    #
    #     repeat_length / sequence / motif_count / A+T%
    #
    # remain available independently of:
    #
    #     mean_core_quality
    #
    # This allows the raw repeat distribution to be analyzed without
    # introducing an unintended quality-based filtering step.
    # =========================================================================
    
    records.append(row)

# =============================================================================
# STEP 23 — Convert the per-read measurements and QC records to DataFrames
# =============================================================================
#
# overlap_df:
#     Main per-read dataset containing all retained nonempty repeat cores.
#
# overlap_qc:
#     QC dataset describing repeat retention and HiFi-quality recovery.
# =============================================================================


overlap_df = pd.DataFrame(records)
overlap_qc = pd.DataFrame(qc_records)

if overlap_df.empty:
    raise ValueError("No overlapping reads with valid nonempty cores.")

# =============================================================================
# STEP 24 — Sort retained reads by repeat length
# =============================================================================
#
# Reads are ordered first by repeat length and then by read ID.
#
# This is especially useful for subsequent sequence-track plots because
# reads with similar repeat sizes will appear together.
# =============================================================================


overlap_df = (
    overlap_df
    .sort_values(["repeat_length", "read_id"])
    .reset_index(drop=True)
)

# =============================================================================
# STEP 25 — Report overall dataset and QC metrics
# =============================================================================
#
# The summary distinguishes several important quantities:
#
# C9orf72 spanning records
#     Total unique primary TRGT records assigned to the locus.
#
# Read IDs also in HiFi BAM
#     Number of those read IDs found in the original HiFi BAM.
#
# Retained nonempty cores
#     Reads for which a valid TRGT-defined core could be generated.
#
# Cores shorter than 6 bp
#     Very short retained cores. Since the canonical C9orf72 motif is 6 bp,
#     these sequences cannot contain one complete 6-bp motif.
#
# "Read status"
#     Describes whether the repeat/core itself was retained.
#
# "HiFi quality recovery"
#     Separately describes whether per-base quality could be successfully
#     recovered for that repeat.
# =============================================================================


print(f"Sample: {sample_name}")
print(f"C9orf72 spanning records: {len(spanning_reads):,}")
print(f"Read IDs also in HiFi BAM: {len(overlap_ids):,}")
print(f"Retained nonempty cores: {len(overlap_df):,}")
print(f"Cores shorter than 6 bp: {(overlap_df.repeat_length < 6).sum():,}")
print("\nRead status:")
print(overlap_qc["status"].value_counts())

print("\nHiFi quality recovery:")
print(overlap_qc["quality_status"].value_counts())

display(
    overlap_df[
        ["read_id", "spanning_length", "repeat_length",
         "at_percentage", "mean_core_quality"]
    ].head(15)
)

In [ ]:
# =============================================================================
# PB* C9orf72 — Single-Molecule Per-Base HiFi Quality Tracks
# =============================================================================

# Visualize the per-base PacBio HiFi Phred quality across the TRGT-defined
# C9orf72 repeat/core region for each individual spanning read.
#
# Each horizontal row in the final heatmap represents ONE molecule/read.
#
# X-axis:
#     Position within the trimmed repeat region (bp), oriented in the
#     gene 5′ → 3′ direction.
#
# Y-axis:
#     Individual HiFi-overlapping reads, sorted by repeat length.
#
# Color:
#     Per-base Phred quality score from the original PacBio HiFi BAM.

# Single-molecule per-base HiFi quality tracks.
# Run after the raw-data metrics cell. Quality arrays are already trimmed
# and oriented to gene 5-prime -> 3-prime by that loader.

# =============================================================================
# STEP 1 — Import packages required for numerical arrays and plotting
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt


# =============================================================================
# STEP 2 — Sort reads by repeat length
# =============================================================================
#
# Sort the retained reads using:
#
#     1. repeat_length
#     2. read_id
#
# Sorting by repeat length makes the final heatmap easier to interpret:
# shorter repeats appear together and progressively longer repeats extend
# farther along the x-axis.
#
# read_id is used as a secondary sorting key to provide deterministic
# ordering among reads with the same repeat length.
# =============================================================================


quality_track_df = overlap_df.sort_values(
    ["repeat_length", "read_id"]
).reset_index(drop=True)
if quality_track_df.empty:
    raise ValueError("No HiFi-overlapping repeat regions available.")

# =============================================================================
# STEP 3 — Determine dimensions of the quality-track matrix
# =============================================================================
#
# track_lengths:
#     Repeat length of every retained read.
#
# n_tracks:
#     Total number of reads that will be represented in the figure.
#
# max_track_length:
#     Length of the longest repeat in the dataset.
#
# Because different molecules can have different repeat lengths, they cannot
# directly be stored in a normal rectangular matrix without padding.
#
# Therefore, the final matrix will have:
#
#       rows    = number of reads
#       columns = longest repeat length
#
# Example:
#
#                         Repeat coordinate
#                   1   2   3   4   5   6   7 ...
#
#     Read 1        Q   Q   Q   Q
#     Read 2        Q   Q   Q   Q   Q   Q
#     Read 3        Q   Q   Q
#     Read 4        Q   Q   Q   Q   Q   Q   Q
#
# Positions beyond the actual length of each read will remain NaN.
# =============================================================================


track_lengths = quality_track_df["repeat_length"].to_numpy(dtype=int)
n_tracks = len(quality_track_df)
max_track_length = int(track_lengths.max())

# =============================================================================
# STEP 4 — Create the per-base quality matrix
# =============================================================================
#
# Initialize a rectangular matrix filled with NaN:
#
#       shape = n_reads × maximum_repeat_length
#
# Each row corresponds to one molecule.
# Each column corresponds to one repeat coordinate.
#
# NaN is intentionally used as the default because positions outside the
# actual repeat length of a read should NOT be interpreted as having a
# quality score of zero.
#
# quality_image therefore becomes the numerical matrix underlying the
# final heatmap.
# =============================================================================

quality_image = np.full((n_tracks, max_track_length), np.nan, dtype=np.float32)
missing_quality_reads = 0

# =============================================================================
# STEP 5 — Add each read's per-base Phred scores to the matrix
# =============================================================================
#
# For each retained read:
#
#     core_qualities = array of per-base Phred scores across the repeat.
#
# For example:
#
#     core_sequence:
#         GGGGCCGGGGCCATGGGGCC...
#
#     core_qualities:
#         35 38 40 39 36 37 21 18 22...
#
# Because core_qualities was generated in the previous cell, the array is
# already:
#
#     • trimmed to the repeat/core;
#     • aligned base-for-base with core_sequence;
#     • oriented in gene 5′ → 3′ direction.
#
# Therefore:
#
#     quality_image[i, j]
#
# represents the Phred quality of base j in molecule/read i.
# =============================================================================

for i, row in quality_track_df.iterrows():
    qualities = row["core_qualities"]

    # =========================================================================
    # STEP 6 — Track reads without recoverable quality
    # =========================================================================
    #
    # Some valid TRGT repeat cores may not have recoverable HiFi quality.
    #
    # These reads were intentionally retained by the raw-data pipeline so
    # that quality recovery does not determine whether a repeat exists in
    # the dataset.
    #
    # If core_qualities is None:
    #
    #     • increment missing_quality_reads;
    #     • leave the corresponding quality_image row as NaN;
    #     • continue to the next read.
    #
    # These reads will later be visible through the gray backbone rather
    # than being assigned artificial quality values.
    # =========================================================================
    
    if qualities is None:
        missing_quality_reads += 1
        continue
    qualities = np.asarray(qualities, dtype=np.float32)

    # =========================================================================
    # STEP 7 — Verify sequence/quality length consistency
    # =========================================================================
    #
    # There should be exactly ONE Phred quality score for every base in the
    # repeat/core.
    #
    # Therefore:
    #
    #     len(core_qualities) == repeat_length
    #
    # must always be true.
    #
    # A mismatch would indicate that sequence trimming/orientation and
    # quality extraction became misaligned in the upstream pipeline.
    # Rather than plotting potentially incorrect data, stop with an error.
    # =========================================================================
    
    if len(qualities) != row["repeat_length"]:
        raise ValueError(f"Sequence/quality length mismatch: {row['read_id']}")


    # =========================================================================
    # STEP 8 — Insert the quality array into the corresponding matrix row
    # =========================================================================
    #
    # Quality scores occupy positions:
    #
    #     0 → repeat_length
    #
    # for this read.
    #
    # All positions after the end of the repeat remain NaN.
    #
    # Therefore, molecules with different repeat lengths can be represented
    # in the same rectangular matrix without adding artificial values.
    # =========================================================================

        
    quality_image[i, :len(qualities)] = qualities

# =============================================================================
# STEP 9 — Initialize the single-molecule quality figure
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 8))
# Gray backbones identify missing quality; space beyond each read is white.
ax.hlines(np.arange(n_tracks) + 0.5, 0, track_lengths,
          color="#D0D0D0", linewidth=0.5, zorder=1)
quality_cmap = plt.get_cmap("viridis").copy()
quality_cmap.set_bad((1, 1, 1, 0))
im = ax.imshow(
    np.ma.masked_invalid(quality_image),
    origin="lower", aspect="auto", interpolation="nearest",
    extent=(0, max_track_length, 0, n_tracks),
    cmap=quality_cmap, vmin=0, vmax=40, rasterized=True, zorder=2,
)
ax.set_xlim(0, max_track_length * 1.02)
ax.set_ylim(0, n_tracks)
ax.set_xlabel("Repeat Coordinate (bp, gene 5′ → 3′)", fontsize=11, fontweight="bold")
ax.set_ylabel(f"HiFi-overlapping Reads (Sorted by Repeat Length, N={n_tracks:,})",
              fontsize=11, fontweight="bold")
ax.set_title(f"{sample_name} {target_locus}: Single-Molecule Per-Base Quality Scores\n"
             "Gene 5′ → 3′; flanks removed", fontsize=12, fontweight="bold", loc="left")
ax.grid(axis="x", linestyle="--", linewidth=0.5, alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
cbar.set_label("Phred Quality Score (Q)", fontsize=10, fontweight="bold")
cbar.set_ticks([0, 10, 20, 30, 40])
cbar.set_ticklabels(["Q0", "Q10", "Q20", "Q30", "Q40+"])
fig.tight_layout()
quality_track_prefix = f"{sample_name}_{target_locus}_HiFi_overlap_per_base_quality_tracks"
fig.savefig(f"{quality_track_prefix}.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{quality_track_prefix}.svg", dpi=300, bbox_inches="tight")
print(f"Per-base quality tracks: {n_tracks:,} reads; "
      f"{missing_quality_reads:,} without recoverable quality (gray tracks).")
plt.show()


In [ ]:
# =============================================================================
# FIGURE A — Distribution of Exact C9orf72 Motif Counts Per Read
# =============================================================================

# A. Exact motif-count distribution
count_dist = (
    overlap_df["motif_count"]
    .value_counts(normalize=True)
    .mul(100)
    .sort_index()
)

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    count_dist.index,
    count_dist.values,
    width=np.maximum(0.6, count_dist.index.to_numpy() * 0.06),
    color="#2b5c8f",
    edgecolor="black",
    linewidth=0.4,
)

ax.set_xscale("symlog", linthresh=10)
ax.set(
    xlabel="Exact motif count (symmetric-log scale; zero included)",
    ylabel="Overlapping reads (%)",
    title=f"{sample_name} C9orf72 motif distribution (n={len(overlap_df):,})",
)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

print(count_dist.sum())


In [ ]:
# =============================================================================
# FIGURE B — A+T Content versus C9orf72 Repeat Length
# =============================================================================

# Examine the relationship between repeat length and A+T sequence content
# across individual C9orf72-spanning reads.
#
# Each point represents ONE retained HiFi-overlapping C9orf72 read.

# B. A+T percentage versus repeat length
at_df = overlap_df.dropna(subset=["at_percentage"])

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    at_df["at_percentage"],
    at_df["repeat_length"],
    s=20, alpha=0.4, edgecolors="none", color="#2b5c8f",
)

ax.set(
    xlabel="A+T content in trimmed repeat (%)",
    ylabel="Repeat length (bp)",
    title=f"{sample_name} C9orf72: HiFi-overlapping reads (n={len(at_df):,})",
)
ax.grid(alpha=0.25, linestyle="--")
fig.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# FIGURE C — A+T Content versus Mean HiFi Quality of the Repeat Region
# =============================================================================
#
# PURPOSE
# -------
# Test whether A+T content within the trimmed C9orf72 repeat/core is
# associated with PacBio HiFi base-call quality.
#
# Each point represents ONE individual C9orf72-spanning read for which both:
#
#     • A+T percentage
#     • repeat-region per-base HiFi quality

# C. A+T percentage versus mean repeat quality
quality_df = overlap_df.dropna(
    subset=["at_percentage", "mean_core_quality"]
)

if quality_df.empty:
    print("No recoverable quality measurements for the quality scatter plot.")
else:
    x = quality_df["at_percentage"].to_numpy()
    y = quality_df["mean_core_quality"].to_numpy()

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.scatter(
        x, y, s=20, alpha=0.35, edgecolors="none",
        color="#2b5c8f", label=f"Reads (n={len(x):,})",
    )

    short = quality_df[quality_df["repeat_length"] < 6]
    if not short.empty:
        ax.scatter(
            short["at_percentage"], short["mean_core_quality"],
            s=85, facecolors="none", edgecolors="red",
            linewidths=1.5, zorder=5,
            label=f"Cores <6 bp (n={len(short)})",
        )

    if len(x) >= 3 and np.ptp(x) > 0 and np.ptp(y) > 0:
        r, p = pearsonr(x, y)
        rho, sp = spearmanr(x, y)
        xx = np.linspace(x.min(), x.max(), 200)

        ax.plot(
            xx, np.polyval(np.polyfit(x, y, 1), xx),
            color="#e27b38", label="Linear trend",
        )
        ax.text(
            0.04, 0.96,
            f"Pearson r={r:.3f}, p={p:.2e}\n"
            f"Spearman ρ={rho:.3f}, p={sp:.2e}",
            transform=ax.transAxes, va="top",
            bbox=dict(facecolor="white", alpha=0.9),
        )

    ax.set(
        xlabel="A+T content in trimmed repeat (%)",
        ylabel="Mean repeat-region HiFi base quality (Phred)",
        title=f"{sample_name} C9orf72: A+T content vs quality",
    )
    ax.legend()
    ax.grid(alpha=0.25, linestyle="--")
    fig.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# FIGURE D — Repeat Length versus Mean Repeat-Region HiFi Base Quality
# =============================================================================
#
# PURPOSE
# -------
# Examine whether the length of the TRGT-defined C9orf72 repeat/core is
# associated with PacBio HiFi base-call quality.
#
# Each point represents ONE individual C9orf72-spanning read for which


# Repeat length versus mean repeat-region HiFi base quality.
# Uses the individual sample loaded in the raw-data metrics cell.
length_quality_df = overlap_df.dropna(
    subset=["repeat_length", "mean_core_quality"]
).copy()
if length_quality_df.empty:
    raise ValueError("No repeat-length / HiFi-quality measurements available.")
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.scatter(
    length_quality_df["repeat_length"],
    length_quality_df["mean_core_quality"],
    s=20, alpha=0.4, color="#2b5c8f", edgecolors="none",
    rasterized=True,
)
ax.set_xlabel("Repeat Length (bp; Flanks Removed)", fontsize=11, fontweight="bold")
ax.set_ylabel("Mean Repeat-Region Base Quality (Phred)", fontsize=11, fontweight="bold")
ax.set_title(
    f"{sample_name} {target_locus}: Repeat Length vs Mean Base Quality\n"
    f"HiFi-overlapping reads (n={len(length_quality_df):,})",
    fontsize=12, fontweight="bold",
)
ax.grid(True, linestyle="--", alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(f"{sample_name}_{target_locus}_HiFi_overlap_length_vs_quality.png",
            dpi=300, bbox_inches="tight")
fig.savefig(f"{sample_name}_{target_locus}_HiFi_overlap_length_vs_quality.svg",
            bbox_inches="tight")
plt.show()


In [ ]:
# =============================================================================
# Exact Motif-Count Distribution
# =============================================================================

# Compare the distribution of exact C9orf72 repeat-motif counts between
# Experimental and control at the single-read level.


from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd
import pysam


# ============================================================
# 1. Sample configuration
# ============================================================

samples = {
    "PB*": {                                              # The experimental group
        "spanning": "*.trgt.sorted.spanning.bam",
        "hifi": "*.hifi.pbmm2.bam",
    },
    "PB*": {                                              # The control group
        "spanning": "*.trgt.sorted.spanning.bam",
        "hifi": "*.hifi.pbmm2.bam",
    },
}

target_locus = "FTDALS1_C9orf72"
target_strand = "-"
fl_size = 50 # since we know the FL is 50 bp 

colors = {
    "PB*": "#2b5c8f",
    "PB*": "#e27b38",
}

trantab = str.maketrans(
    "ACGTNacgtn",
    "TGCANtgcan",
)


def revcomp(seq):
    return seq.translate(trantab)[::-1]


# ============================================================
# 2. Extract motif counts from HiFi-overlapping spanning reads
# ============================================================

def extract_motif_counts(
    spanning_bam,
    hifi_bam,
    locus_name,
    strand="-",
    fl_size=50,
):
    for filename in (spanning_bam, hifi_bam):
        if not Path(filename).is_file():
            raise FileNotFoundError(filename)

    # A. Collect unique primary spanning records at C9orf72.
    spanning_sequences = {}

    with pysam.AlignmentFile(
        spanning_bam, "rb", check_sq=False
    ) as bam:
        for read in bam.fetch(until_eof=True):
            if not read.has_tag("TR"):
                continue
            if read.get_tag("TR") != locus_name:
                continue
            if read.is_secondary or read.is_supplementary:
                continue

            if read.query_name in spanning_sequences:
                raise ValueError(
                    f"Duplicate primary spanning record: {read.query_name}"
                )

            spanning_sequences[read.query_name] = read.query_sequence

    # B. Find exact read-ID overlap with the HiFi BAM.
    # No base-quality, read-quality, or mapping-quality threshold.
    overlap_ids = set()

    with pysam.AlignmentFile(
        hifi_bam, "rb", check_sq=False
    ) as bam:
        for read in bam.fetch(until_eof=True):
            if read.query_name in spanning_sequences:
                overlap_ids.add(read.query_name)

    # C. Calculate motif counts using the spanning sequences.
    records = []
    missing_sequence = 0
    empty_core = 0

    for read_id in sorted(overlap_ids):
        seq = spanning_sequences[read_id]

        if not seq:
            missing_sequence += 1
            continue

        seq = seq.upper()

        # Same gene-direction convention as your original code.
        if strand == "-":
            seq = revcomp(seq)

        # Remove exactly 50 bp from each end.
        # Retain all nonempty cores, including 1–5 bp.
        if len(seq) <= 2 * fl_size:
            empty_core += 1
            continue

        core_seq = seq[fl_size:len(seq) - fl_size]

        # Same exact-motif counting logic as your original code.
        count_fwd = len(re.findall(r"(?=GGGGCC)", core_seq))
        count_rev = len(re.findall(r"(?=GGCCCC)", core_seq))

        records.append({
            "read_id": read_id,
            "spanning_length": len(seq),
            "repeat_length": len(core_seq),
            "count_fwd": count_fwd,
            "count_rev": count_rev,
            "motif_count": max(count_fwd, count_rev),
        })

    columns = [
        "read_id",
        "spanning_length",
        "repeat_length",
        "count_fwd",
        "count_rev",
        "motif_count",
    ]
    result = pd.DataFrame(records, columns=columns)

    print(f"  C9orf72 spanning reads: {len(spanning_sequences):,}")
    print(f"  Also present in HiFi BAM: {len(overlap_ids):,}")
    print(f"  Missing sequence: {missing_sequence:,}")
    print(f"  No core after flank removal: {empty_core:,}")
    print(f"  Reads retained: {len(result):,}")

    if not result.empty:
        print(
            "  Retained cores shorter than 6 bp: "
            f"{(result['repeat_length'] < 6).sum():,}"
        )
        print(
            "  Reads with zero exact motifs: "
            f"{(result['motif_count'] == 0).sum():,}"
        )

    return result


# ============================================================
# 3. Process samples
# ============================================================

data_dist = {}
sample_sizes = {}
per_read_data = {}

for name, paths in samples.items():
    print(f"\nProcessing {name}...")

    read_df = extract_motif_counts(
        spanning_bam=paths["spanning"],
        hifi_bam=paths["hifi"],
        locus_name=target_locus,
        strand=target_strand,
        fl_size=fl_size,
    )

    per_read_data[name] = read_df
    sample_sizes[name] = len(read_df)

    if not read_df.empty:
        # Denominator includes every retained read, even zero-motif reads.
        data_dist[name] = (
            read_df["motif_count"]
            .value_counts(normalize=True)
            .mul(100)
            .sort_index()
        )
    else:
        data_dist[name] = pd.Series(dtype=float)


# ============================================================
# 4. Two-panel subplot
# ============================================================

fig, axes = plt.subplots(
    2,
    1,
    figsize=(9, 7),
    sharex=True,
    sharey=True,
)

for ax, name in zip(axes, samples):
    dist = data_dist[name]

    # Logarithmic axes can display only positive motif counts.
    positive_dist = dist[dist.index > 0]
    zero_percentage = float(dist.get(0, 0.0))

    ax.bar(
        positive_dist.index,
        positive_dist.values,
        width=positive_dist.index.to_numpy() * 0.08,
        color=colors.get(name, "#333333"),
        alpha=0.8,
        edgecolor="black",
        linewidth=0.5,
    )

    ax.set_xscale("log")
    ax.set_ylabel(
        "HiFi-overlapping Spanning Reads (%)",
        fontsize=10,
        fontweight="bold",
    )
    ax.set_title(
        f"{name} (n={sample_sizes[name]:,})",
        fontsize=11,
        loc="left",
    )

    if sample_sizes[name] == 0:
        annotation = "No usable overlapping reads"
    else:
        annotation = (
            f"Zero exact motifs: {zero_percentage:.2f}%\n"
            "Included in denominator; not shown on log axis"
        )

    ax.text(
        0.98,
        0.95,
        annotation,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=8,
        bbox={
            "facecolor": "white",
            "edgecolor": "#CCCCCC",
            "alpha": 0.9,
        },
    )

    ax.grid(
        True,
        linestyle="--",
        alpha=0.4,
    )
    ax.set_axisbelow(True)

# Provide valid log-axis limits even if no positive counts exist.
positive_counts = [
    int(count)
    for dist in data_dist.values()
    for count in dist.index
    if count > 0
]

if positive_counts:
    axes[-1].set_xlim(
        min(positive_counts) * 0.8,
        max(positive_counts) * 1.2,
    )
else:
    axes[-1].set_xlim(0.8, 10)

axes[-1].set_xlabel(
    "Exact Motif Count (GGGGCC / GGCCCC, Log Scale)",
    fontsize=11,
    fontweight="bold",
)

fig.suptitle(
    f"{target_locus} Repeat Distribution: PB* vs PB*\n"
    "Spanning reads overlapping the corresponding HiFi BAM",
    fontsize=12,
    fontweight="bold",
)

plt.tight_layout(rect=[0, 0, 1, 0.94])

plt.savefig(
    f"{target_locus}_PB*_vs_PB*_HiFi_overlap_subplots.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# Follow with the last cell: 
# Merge two figures: 

import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Uses the samples already present in data_dist.
plot_order = list(data_dist.keys())

fig, ax = plt.subplots(figsize=(10, 6))

ax_inset = inset_axes(
    ax,
    width="47%",
    height="45%",
    loc="upper right",
    bbox_to_anchor=(0, 0, 0.97, 0.95),
    bbox_transform=ax.transAxes,
)

zero_notes = []
positive_counts = []

for name in plot_order:
    dist = data_dist[name].sort_index()

    # Zero cannot be displayed on a logarithmic X-axis.
    # Preserve the original percentages without renormalizing.
    positive_dist = dist[dist.index > 0]

    if positive_dist.empty:
        zero_notes.append(f"{name}: no positive motif counts")
        continue

    positive_counts.extend(positive_dist.index.tolist())

    color = colors.get(name, "#2b5c8f")
    label = f"{name} (n={sample_sizes[name]:,})"

    # Main plot: transparent bars allow overlap to remain visible.
    ax.bar(
        positive_dist.index,
        positive_dist.values,
        width=positive_dist.index.to_numpy() * 0.08,
        color=color,
        edgecolor=color,
        linewidth=0.7,
        alpha=0.45,
        label=label,
    )

    # Inset: same distributions, enlarged low-frequency range.
    ax_inset.bar(
        positive_dist.index,
        positive_dist.values,
        width=positive_dist.index.to_numpy() * 0.08,
        color=color,
        edgecolor=color,
        linewidth=0.7,
        alpha=0.45,
    )

    zero_percentage = float(dist.get(0, 0.0))
    if zero_percentage > 0:
        zero_notes.append(
            f"{name}: zero motifs = {zero_percentage:.2f}%"
        )

ax.set_xscale("log")
ax_inset.set_xscale("log")

if positive_counts:
    x_limits = (
        min(positive_counts) * 0.8,
        max(positive_counts) * 1.2,
    )
else:
    x_limits = (0.8, 10)

ax.set_xlim(x_limits)
ax_inset.set_xlim(x_limits)

ax.set_ylim(bottom=0)
ax_inset.set_ylim(0, 2)

ax.set_xlabel(
    "Exact Motif Count (GGGGCC / GGCCCC, Log Scale)",
    fontsize=11,
    fontweight="bold",
)
ax.set_ylabel(
    "HiFi-overlapping Spanning Reads (%)",
    fontsize=11,
    fontweight="bold",
)
ax.set_title(
    f"{target_locus}: {' vs '.join(plot_order)}",
    fontsize=12,
    fontweight="bold",
    loc="left",
)

ax.legend(loc="upper left", frameon=True, fontsize=10)
ax.grid(True, linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

ax_inset.set_title("Zoom-in (Y ≤ 2%)", fontsize=9, pad=4)
ax_inset.tick_params(axis="both", which="major", labelsize=8)
ax_inset.grid(True, linestyle=":", alpha=0.4)
ax_inset.set_axisbelow(True)

if zero_notes:
    fig.text(
        0.12,
        0.02,
        "; ".join(zero_notes)
        + "\nZero-motif reads remain in the denominator; "
          "zero is not shown on the log axis.",
        fontsize=8,
    )

# Manual spacing avoids tight_layout warnings with inset_axes.
fig.subplots_adjust(
    left=0.12,
    right=0.97,
    top=0.90,
    bottom=0.19 if zero_notes else 0.13,
)

sample_label = "_vs_".join(plot_order)

fig.savefig(
    f"{target_locus}_{sample_label}_HiFi_overlap_overlaid_inset.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# Methylation 

# =============================================================================
# Import required Python packages
# =============================================================================
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import pysam
from IPython.display import display


# ============================================================
# 1. Configuration
# ============================================================

data_dir = Path.cwd()

sample_name = "PB*"
target_locus = "FTDALS1_C9orf72"
target_strand = "-"

spanning_bam = data_dir / (
    "*.trgt.sorted.spanning.bam"
)
hifi_bam = data_dir / "PB*.hifi.pbmm2.bam"

output_prefix = data_dir / (
    f"{sample_name}_{target_locus}_HiFi_overlap"
)

# Plot all successfully mapped, nonempty repeat regions.
# No minimum repeat length other than >0.
# No read-quality or ML-score threshold.

for path in (spanning_bam, hifi_bam):
    if not path.is_file():
        raise FileNotFoundError(path)


complement = str.maketrans(
    "ACGTRYMKBDHVN",
    "TGCAYRKMVHDBN",
)


def reverse_complement(sequence):
    return sequence.translate(complement)[::-1]


def find_unique_span(full_sequence, span_sequence):
    """Locate the full spanning sequence in either orientation."""
    hits = []

    for orientation, needle in [
        ("forward", span_sequence),
        ("reverse", reverse_complement(span_sequence)),
    ]:
        position = full_sequence.find(needle)

        while position >= 0:
            hits.append((position, orientation))

            if len(hits) > 1:
                return None, "ambiguous_sequence_match"

            position = full_sequence.find(needle, position + 1)

    if not hits:
        return None, "sequence_not_found"

    return hits[0], "success"


# ============================================================
# 2. Load C9orf72 spanning reads
# ============================================================

spanning_reads = {}

with pysam.AlignmentFile(
    spanning_bam, "rb", check_sq=False
) as bam:
    for read in bam.fetch(until_eof=True):
        if not read.has_tag("TR"):
            continue

        if read.get_tag("TR") != target_locus:
            continue

        if read.is_secondary or read.is_supplementary:
            continue

        if read.query_name in spanning_reads:
            raise ValueError(
                f"Duplicate primary spanning record: {read.query_name}"
            )

        spanning_reads[read.query_name] = read

if not spanning_reads:
    raise ValueError(f"No spanning reads found for {target_locus}.")

target_ids = set(spanning_reads)


# ============================================================
# 3. Identify HiFi overlap and recover primary HiFi records
# ============================================================

overlap_ids = set()
original_reads = {}

with pysam.AlignmentFile(
    hifi_bam, "rb", check_sq=False
) as bam:
    for read in bam.fetch(until_eof=True):
        if read.query_name not in target_ids:
            continue

        # Membership is based on exact read ID.
        overlap_ids.add(read.query_name)

        # Use primary records to extract sequence and MM/ML data.
        if read.is_secondary or read.is_supplementary:
            continue

        if read.query_name in original_reads:
            raise ValueError(
                f"Duplicate primary HiFi record: {read.query_name}"
            )

        original_reads[read.query_name] = read

print(f"Sample: {sample_name}")
print(f"C9orf72 spanning reads: {len(spanning_reads):,}")
print(f"Read IDs also present in HiFi BAM: {len(overlap_ids):,}")
print(f"Primary HiFi records recovered: {len(original_reads):,}")


# ============================================================
# 4. Map repeat coordinates and extract C+m calls
# ============================================================

summary = []
modification_records = []

for read_id, span_read in spanning_reads.items():
    record = {
        "read_id": read_id,
        "source": "hifi",
        "status": None,
        "orientation": None,
        "left_flank": None,
        "right_flank": None,
        "repeat_start": None,
        "repeat_end": None,
        "repeat_length": None,
        "mod_data_status": "not_processed",
        "n_repeat_5mc_strand0": 0,
        "n_repeat_5mc_strand1": 0,
        "n_repeat_5mc_total": 0,
        "n_unknown_ML": 0,
    }

    if read_id not in overlap_ids:
        record["status"] = "not_in_hifi"
        summary.append(record)
        continue

    original = original_reads.get(read_id)

    if original is None:
        record["status"] = "no_primary_hifi_record"
        summary.append(record)
        continue

    if not span_read.query_sequence or not original.query_sequence:
        record["status"] = "missing_sequence"
        summary.append(record)
        continue

    span_sequence = span_read.query_sequence.upper()
    original_sequence = original.query_sequence.upper()

    # FL refers to the stored spanning sequence.
    try:
        flanks = span_read.get_tag("FL")

        if len(flanks) != 2:
            raise ValueError

        left_flank, right_flank = map(int, flanks)

        if min(left_flank, right_flank) < 0:
            raise ValueError

    except (KeyError, TypeError, ValueError):
        record["status"] = "missing_or_invalid_FL"
        summary.append(record)
        continue

    repeat_length = (
        len(span_sequence) - left_flank - right_flank
    )

    record.update({
        "left_flank": left_flank,
        "right_flank": right_flank,
        "repeat_length": repeat_length,
    })

    # Keep 1-bp cores. Only empty/invalid regions are excluded.
    if repeat_length <= 0:
        record["status"] = "empty_or_invalid_core"
        summary.append(record)
        continue

    # Match the full spanning sequence, including flanks.
    match, match_status = find_unique_span(
        original_sequence,
        span_sequence,
    )

    if match is None:
        record["status"] = match_status
        summary.append(record)
        continue

    match_position, orientation = match

    if orientation == "forward":
        repeat_start = match_position + left_flank
        repeat_end = (
            match_position + len(span_sequence) - right_flank
        )
    else:
        repeat_start = match_position + right_flank
        repeat_end = (
            match_position + len(span_sequence) - left_flank
        )

    reference_core = span_sequence[
        left_flank:len(span_sequence) - right_flank
    ]

    expected_core = (
        reference_core
        if orientation == "forward"
        else reverse_complement(reference_core)
    )

    assert original_sequence[repeat_start:repeat_end] == expected_core
    assert repeat_end - repeat_start == repeat_length

    record.update({
        "status": "success",
        "orientation": orientation,
        "repeat_start": repeat_start,
        "repeat_end": repeat_end,
    })

    # Missing modification information does not remove the read.
    mm_tag = next(
        (tag for tag in ("MM", "Mm") if original.has_tag(tag)),
        None,
    )
    ml_tag = next(
        (tag for tag in ("ML", "Ml") if original.has_tag(tag)),
        None,
    )

    if mm_tag is None or ml_tag is None:
        record["mod_data_status"] = "missing_MM_or_ML"
        summary.append(record)
        continue

    mm_values = original.get_tag(mm_tag)
    ml_values = original.get_tag(ml_tag)

    # Guard against empty modification tags.
    if not mm_values or ml_values is None or len(ml_values) == 0:
        record["mod_data_status"] = "empty_MM_or_ML"
        summary.append(record)
        continue

    try:
        modified = original.modified_bases
    except (ValueError, TypeError, OverflowError):
        record["mod_data_status"] = "modification_decode_error"
        summary.append(record)
        continue

    if modified is None:
        record["mod_data_status"] = "modification_decode_unavailable"
        summary.append(record)
        continue

    record["mod_data_status"] = "decoded"

    # Keep both C+m modification strands.
    for mod_strand in (0, 1):
        sites = modified.get(("C", mod_strand, "m"), [])

        for read_position, ml_score in sites:
            if not (repeat_start <= read_position < repeat_end):
                continue

            # Original HiFi coordinates -> spanning-sequence direction.
            if orientation == "forward":
                reference_position = read_position - repeat_start
                reference_mod_strand = mod_strand
            else:
                reference_position = repeat_end - 1 - read_position
                reference_mod_strand = 1 - mod_strand

            # Spanning direction -> gene direction, as in your notebook.
            if target_strand == "-":
                gene_position = (
                    repeat_length - 1 - reference_position
                )
                gene_mod_strand = 1 - reference_mod_strand
            else:
                gene_position = reference_position
                gene_mod_strand = reference_mod_strand

            modification_records.append({
                "read_id": read_id,
                "source": "hifi",
                "orientation": orientation,
                "repeat_length": repeat_length,
                "repeat_relative_pos": gene_position,
                "original_read_pos": read_position,
                "mod_strand_original": mod_strand,
                "mod_strand_standardized": gene_mod_strand,
                "ML_score": ml_score,
            })

            record[f"n_repeat_5mc_strand{mod_strand}"] += 1
            record["n_repeat_5mc_total"] += 1

            if ml_score < 0:
                record["n_unknown_ML"] += 1

    summary.append(record)


# ============================================================
# 5. DataFrames and QC
# ============================================================

summary_df = pd.DataFrame(summary)

meth_columns = [
    "read_id",
    "source",
    "orientation",
    "repeat_length",
    "repeat_relative_pos",
    "original_read_pos",
    "mod_strand_original",
    "mod_strand_standardized",
    "ML_score",
]

meth_df = pd.DataFrame(
    modification_records,
    columns=meth_columns,
)

order_df = (
    summary_df.loc[summary_df["status"] == "success"]
    .sort_values(["repeat_length", "read_id"])
    .reset_index(drop=True)
)

print("\nRead-processing status:")
print(summary_df["status"].value_counts())

if order_df.empty:
    summary_df.to_csv(f"{output_prefix}_QC.csv", index=False)
    raise ValueError(
        "No overlapping reads could be mapped to valid repeat coordinates."
    )

order_df["repeat_length"] = order_df["repeat_length"].astype(int)

print("\nModification-data status:")
print(order_df["mod_data_status"].value_counts())

print(f"\nReads retained for tracks: {len(order_df):,}")
print(
    "Retained cores shorter than 6 bp: "
    f"{(order_df['repeat_length'] < 6).sum():,}"
)
print(f"Repeat-internal C+m entries: {len(meth_df):,}")

assert (order_df["repeat_length"] > 0).all()

if not meth_df.empty:
    assert (meth_df["repeat_relative_pos"] >= 0).all()
    assert (
        meth_df["repeat_relative_pos"] < meth_df["repeat_length"]
    ).all()

display(
    order_df[
        [
            "read_id",
            "repeat_length",
            "orientation",
            "mod_data_status",
            "n_repeat_5mc_total",
        ]
    ].head(15)
)

summary_df.to_csv(f"{output_prefix}_QC.csv", index=False)
meth_df.to_csv(f"{output_prefix}_5mC_sites.csv", index=False)


# ============================================================
# 6. Single-molecule 5mC tracks
# ============================================================

n_reads = len(order_df)

read_to_y = {
    read_id: index + 0.5
    for index, read_id in enumerate(order_df["read_id"])
}

fig, ax = plt.subplots(figsize=(14, 9))

# Every successfully mapped read gets a backbone.
# Missing modification calls do not imply absence of methylation.
ax.hlines(
    y=np.arange(n_reads) + 0.5,
    xmin=0,
    xmax=order_df["repeat_length"].to_numpy(),
    color="#BDBDBD",
    linewidth=0.5,
    alpha=0.4,
    zorder=1,
)

norm = mcolors.Normalize(vmin=0, vmax=255)
cmap = plt.get_cmap("coolwarm")

if not meth_df.empty:
    plot_df = meth_df.copy()
    plot_df["y"] = plot_df["read_id"].map(read_to_y)

    # Negative scores denote unavailable ML, not low methylation.
    # Retain them in the table, but do not assign a numeric color.
    scored_sites = plot_df[
        plot_df["ML_score"].between(0, 255)
    ]
    unknown_sites = plot_df[plot_df["ML_score"] < 0]

    ax.scatter(
        scored_sites["repeat_relative_pos"],
        scored_sites["y"],
        c=scored_sites["ML_score"],
        cmap=cmap,
        norm=norm,
        marker="|",
        s=8,
        linewidths=0.5,
        rasterized=True,
        zorder=2,
    )

    if not unknown_sites.empty:
        ax.scatter(
            unknown_sites["repeat_relative_pos"],
            unknown_sites["y"],
            color="#555555",
            marker="x",
            s=8,
            linewidths=0.5,
            rasterized=True,
            label="ML unavailable",
            zorder=3,
        )
        ax.legend(loc="lower right")

color_mapper = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
color_mapper.set_array([])

cbar = fig.colorbar(
    color_mapper,
    ax=ax,
    pad=0.02,
    fraction=0.035,
)
cbar.set_label("5mC ML score")
cbar.set_ticks([0, 64, 128, 192, 255])

ax.set_xlim(
    0,
    max(1, order_df["repeat_length"].max()) * 1.02,
)
ax.set_ylim(0, n_reads)

ax.set_xlabel(
    "Repeat coordinate (bp, gene 5′ → 3′)",
    fontsize=12,
    fontweight="bold",
)
ax.set_ylabel(
    f"HiFi-overlapping reads sorted by repeat length (n={n_reads:,})",
    fontsize=12,
    fontweight="bold",
)
ax.set_title(
    f"{sample_name} {target_locus}: Single-Molecule 5mC Tracks\n"
    "Both modification strands; no ML threshold",
    fontsize=13,
    fontweight="bold",
    loc="left",
)

ax.grid(
    axis="x",
    linestyle="--",
    linewidth=0.5,
    alpha=0.2,
)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

fig.savefig(
    f"{output_prefix}_5mC_tracks.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    f"{output_prefix}_5mC_tracks.svg",
    bbox_inches="tight",
)

plt.show()

In [ ]:
# =============================================================================
# C9orf72 — Single-Molecule 5mC in the 2-kb Upstream Region
# =============================================================================
def run_als_upstream_2kb():
    from pathlib import Path

    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    import numpy as np
    import pandas as pd
    import pysam
    from IPython.display import display


    # ============================================================
    # 1. Configuration
    # ============================================================

    data_dir = Path('')
    upstream_bp = 2000
    output_dir = Path('')
    output_dir.mkdir(parents=True, exist_ok=True)

    sample_name = "PB*" # The ALS one 
    target_locus = "FTDALS1_C9orf72"
    target_strand = "-"

    spanning_bam = data_dir / (
        "*.trgt.sorted.spanning.bam"
    )
    hifi_bam = data_dir / "PB*.hifi.pbmm2.bam"

    output_prefix = output_dir / (
        f"{sample_name}_{target_locus}_HiFi_upstream_2kb"
    )

    # Plot all successfully mapped, nonempty repeat regions.
    # No minimum repeat length other than >0.
    # No read-quality or ML-score threshold.

    for path in (spanning_bam, hifi_bam):
        if not path.is_file():
            raise FileNotFoundError(path)

    complement = str.maketrans(
        "ACGTRYMKBDHVN",
        "TGCAYRKMVHDBN",
    )


    def reverse_complement(sequence):
        return sequence.translate(complement)[::-1]


    def find_unique_span(full_sequence, span_sequence):
        """Locate the full spanning sequence in either orientation."""
        hits = []

        for orientation, needle in [
            ("forward", span_sequence),
            ("reverse", reverse_complement(span_sequence)),
        ]:
            position = full_sequence.find(needle)

            while position >= 0:
                hits.append((position, orientation))

                if len(hits) > 1:
                    return None, "ambiguous_sequence_match"

                position = full_sequence.find(needle, position + 1)

        if not hits:
            return None, "sequence_not_found"

        return hits[0], "success"


    # ============================================================
    # 2. Load C9orf72 spanning reads
    # ============================================================

    spanning_reads = {}

    with pysam.AlignmentFile(
        spanning_bam, "rb", check_sq=False
    ) as bam:
        for read in bam.fetch(until_eof=True):
            if not read.has_tag("TR"):
                continue

            if read.get_tag("TR") != target_locus:
                continue

            if read.is_secondary or read.is_supplementary:
                continue

            if read.query_name in spanning_reads:
                raise ValueError(
                    f"Duplicate primary spanning record: {read.query_name}"
                )

            spanning_reads[read.query_name] = read

    if not spanning_reads:
        raise ValueError(f"No spanning reads found for {target_locus}.")

    target_ids = set(spanning_reads)


    # ============================================================
    # 3. Identify HiFi overlap and recover primary HiFi records
    # ============================================================

    overlap_ids = set()
    original_reads = {}

    with pysam.AlignmentFile(
        hifi_bam, "rb", check_sq=False
    ) as bam:
        for read in bam.fetch(until_eof=True):
            if read.query_name not in target_ids:
                continue

            # Membership is based on exact read ID.
            overlap_ids.add(read.query_name)

            # Use primary records to extract sequence and MM/ML data.
            if read.is_secondary or read.is_supplementary:
                continue

            if read.query_name in original_reads:
                raise ValueError(
                    f"Duplicate primary HiFi record: {read.query_name}"
                )

            original_reads[read.query_name] = read

    print(f"Sample: {sample_name}")
    print(f"C9orf72 spanning reads: {len(spanning_reads):,}")
    print(f"Read IDs also present in HiFi BAM: {len(overlap_ids):,}")
    print(f"Primary HiFi records recovered: {len(original_reads):,}")


    # ============================================================
    # 4. Anchor the upstream window at the repeat and extract C+m calls
    # ============================================================

    summary = []
    modification_records = []

    for read_id, span_read in spanning_reads.items():
        record = {
            "read_id": read_id,
            "source": "hifi",
            "status": None,
            "orientation": None,
            "left_flank": None,
            "right_flank": None,
            "repeat_start": None,
            "repeat_end": None,
            "repeat_length": None,
            "mod_data_status": "not_processed",
            "n_upstream_5mc_strand0": 0,
            "n_upstream_5mc_strand1": 0,
            "n_upstream_5mc_total": 0,
            "n_unknown_ML": 0,
        }

        if read_id not in overlap_ids:
            record["status"] = "not_in_hifi"
            summary.append(record)
            continue

        original = original_reads.get(read_id)

        if original is None:
            record["status"] = "no_primary_hifi_record"
            summary.append(record)
            continue

        if not span_read.query_sequence or not original.query_sequence:
            record["status"] = "missing_sequence"
            summary.append(record)
            continue

        span_sequence = span_read.query_sequence.upper()
        original_sequence = original.query_sequence.upper()

        # FL refers to the stored spanning sequence.
        try:
            flanks = span_read.get_tag("FL")

            if len(flanks) != 2:
                raise ValueError

            left_flank, right_flank = map(int, flanks)

            if min(left_flank, right_flank) < 0:
                raise ValueError

        except (KeyError, TypeError, ValueError):
            record["status"] = "missing_or_invalid_FL"
            summary.append(record)
            continue

        repeat_length = (
            len(span_sequence) - left_flank - right_flank
        )

        record.update({
            "left_flank": left_flank,
            "right_flank": right_flank,
            "repeat_length": repeat_length,
        })

        # Keep 1-bp cores. Only empty/invalid regions are excluded.
        if repeat_length <= 0:
            record["status"] = "empty_or_invalid_core"
            summary.append(record)
            continue

        # Match the full spanning sequence, including flanks.
        match, match_status = find_unique_span(
            original_sequence,
            span_sequence,
        )

        if match is None:
            record["status"] = match_status
            summary.append(record)
            continue

        match_position, orientation = match

        if orientation == "forward":
            repeat_start = match_position + left_flank
            repeat_end = (
                match_position + len(span_sequence) - right_flank
            )
        else:
            repeat_start = match_position + right_flank
            repeat_end = (
                match_position + len(span_sequence) - left_flank
            )

        reference_core = span_sequence[
            left_flank:len(span_sequence) - right_flank
        ]

        expected_core = (
            reference_core
            if orientation == "forward"
            else reverse_complement(reference_core)
        )

        assert original_sequence[repeat_start:repeat_end] == expected_core
        assert repeat_end - repeat_start == repeat_length

        record.update({
            "status": "success",
            "orientation": orientation,
            "repeat_start": repeat_start,
            "repeat_end": repeat_end,
        })

        # The stored spanning sequence follows the notebook's reference-direction
        # convention. A reverse exact match flips it relative to the HiFi sequence.
        # Gene direction flips once more for a minus-strand locus.
        gene_forward_in_hifi = (orientation == "forward") == (target_strand == "+")
        if gene_forward_in_hifi:
            upstream_start = max(0, repeat_start - upstream_bp)
            upstream_end = repeat_start
        else:
            upstream_start = repeat_end
            upstream_end = min(len(original_sequence), repeat_end + upstream_bp)
        covered_bp = upstream_end - upstream_start
        upstream_sequence = original_sequence[upstream_start:upstream_end]
        if not gene_forward_in_hifi:
            upstream_sequence = reverse_complement(upstream_sequence)
        record.update({
            "upstream_start_hifi_0based": upstream_start,
            "upstream_end_hifi_exclusive": upstream_end,
            "upstream_covered_bp": covered_bp,
            "full_2kb_coverage": covered_bp == upstream_bp,
            "upstream_sequence_gene_5to3": upstream_sequence,
        })

        # Missing modification information does not remove the read.
        mm_tag = next(
            (tag for tag in ("MM", "Mm") if original.has_tag(tag)),
            None,
        )
        ml_tag = next(
            (tag for tag in ("ML", "Ml") if original.has_tag(tag)),
            None,
        )

        if mm_tag is None or ml_tag is None:
            record["mod_data_status"] = "missing_MM_or_ML"
            summary.append(record)
            continue

        mm_values = original.get_tag(mm_tag)
        ml_values = original.get_tag(ml_tag)

        # Guard against empty modification tags.
        if not mm_values or ml_values is None or len(ml_values) == 0:
            record["mod_data_status"] = "empty_MM_or_ML"
            summary.append(record)
            continue

        try:
            modified = original.modified_bases
        except (ValueError, TypeError, OverflowError):
            record["mod_data_status"] = "modification_decode_error"
            summary.append(record)
            continue

        if modified is None:
            record["mod_data_status"] = "modification_decode_unavailable"
            summary.append(record)
            continue

        record["mod_data_status"] = "decoded"

        # Keep both C+m modification strands.
        for mod_strand in (0, 1):
            sites = modified.get(("C", mod_strand, "m"), [])

            for read_position, ml_score in sites:
                if not (upstream_start <= read_position < upstream_end):
                    continue

                # Original HiFi coordinates -> spanning-sequence direction.
                if orientation == "forward":
                    reference_position = read_position - repeat_start
                    reference_mod_strand = mod_strand
                else:
                    reference_position = repeat_end - 1 - read_position
                    reference_mod_strand = 1 - mod_strand

                # Spanning direction -> gene direction, as in your notebook.
                if target_strand == "-":
                    gene_position = (
                        repeat_length - 1 - reference_position
                    )
                    gene_mod_strand = 1 - reference_mod_strand
                else:
                    gene_position = reference_position
                    gene_mod_strand = reference_mod_strand

                modification_records.append({
                    "read_id": read_id,
                    "source": "hifi",
                    "orientation": orientation,
                    "repeat_length": repeat_length,
                    "upstream_relative_pos": gene_position,
                    "original_read_pos": read_position,
                    "mod_strand_original": mod_strand,
                    "mod_strand_standardized": gene_mod_strand,
                    "ML_score": ml_score,
                })

                record[f"n_upstream_5mc_strand{mod_strand}"] += 1
                record["n_upstream_5mc_total"] += 1

                if ml_score < 0:
                    record["n_unknown_ML"] += 1

        summary.append(record)


    # ============================================================
    # 5. DataFrames and QC
    # ============================================================

    summary_df = pd.DataFrame(summary)

    meth_columns = [
        "read_id",
        "source",
        "orientation",
        "repeat_length",
        "upstream_relative_pos",
        "original_read_pos",
        "mod_strand_original",
        "mod_strand_standardized",
        "ML_score",
    ]

    meth_df = pd.DataFrame(
        modification_records,
        columns=meth_columns,
    )

    order_df = (
        summary_df.loc[summary_df["status"] == "success"]
        .sort_values(["repeat_length", "read_id"])
        .reset_index(drop=True)
    )

    print("\nRead-processing status:")
    print(summary_df["status"].value_counts())

    if order_df.empty:
        summary_df.to_csv(f"{output_prefix}_QC.csv", index=False)
        raise ValueError(
            "No overlapping reads could be mapped to valid repeat coordinates."
        )

    order_df["repeat_length"] = order_df["repeat_length"].astype(int)

    print("\nModification-data status:")
    print(order_df["mod_data_status"].value_counts())

    print(f"\nReads retained for tracks: {len(order_df):,}")
    print(
        "Retained cores shorter than 6 bp: "
        f"{(order_df['repeat_length'] < 6).sum():,}"
    )
    print(f"Upstream C+m entries: {len(meth_df):,}")

    assert (order_df["repeat_length"] > 0).all()

    if not meth_df.empty:
        assert (meth_df["upstream_relative_pos"] >= -upstream_bp).all()
        assert (
            meth_df["upstream_relative_pos"] < 0
        ).all()

    display(
        order_df[
            [
                "read_id",
                "repeat_length",
                "orientation",
                "mod_data_status",
                "n_upstream_5mc_total",
            ]
        ].head(15)
    )

    print(f"Full 2 kb coverage: {order_df['full_2kb_coverage'].sum():,}")
    print(f"Partial upstream coverage: {order_df['upstream_covered_bp'].between(1, upstream_bp-1).sum():,}")
    print(f"No upstream sequence available: {(order_df['upstream_covered_bp'] == 0).sum():,}")
    # Unknown ML is preserved; no unreported cytosines are imputed as unmethylated.
    meth_df["5mC_probability_bin_midpoint"] = np.where(
        meth_df["ML_score"].between(0, 255), (meth_df["ML_score"] + 0.5) / 256, np.nan
    )
    summary_df.to_csv(f"{output_prefix}_QC.csv", index=False)
    with open(f"{output_prefix}_sequences.fasta", "w") as fasta:
        for row in order_df.itertuples():
            if row.upstream_covered_bp:
                fasta.write(f">{row.read_id} upstream=-{row.upstream_covered_bp}..-1 repeat_length={row.repeat_length}\n")
                seq = row.upstream_sequence_gene_5to3
                fasta.write("\n".join(seq[i:i+80] for i in range(0, len(seq), 80)) + "\n")
    meth_df.to_csv(f"{output_prefix}_5mC_sites.csv", index=False)


    # ============================================================
    # 6. Single-molecule 5mC tracks
    # ============================================================

    n_reads = len(order_df)

    read_to_y = {
        read_id: index + 0.5
        for index, read_id in enumerate(order_df["read_id"])
    }

    fig, ax = plt.subplots(figsize=(14, 9))

    # Every successfully mapped read gets a backbone.
    # Missing modification calls do not imply absence of methylation.
    ax.hlines(
        y=np.arange(n_reads) + 0.5,
        xmin=-order_df["upstream_covered_bp"].to_numpy(),
        xmax=0,
        color="#BDBDBD",
        linewidth=0.5,
        alpha=0.4,
        zorder=1,
    )

    norm = mcolors.Normalize(vmin=0, vmax=255)
    cmap = plt.get_cmap("coolwarm")

    if not meth_df.empty:
        plot_df = meth_df.copy()
        plot_df["y"] = plot_df["read_id"].map(read_to_y)

        # Negative scores denote unavailable ML, not low methylation.
        # Retain them in the table, but do not assign a numeric color.
        scored_sites = plot_df[
            plot_df["ML_score"].between(0, 255)
        ]
        unknown_sites = plot_df[plot_df["ML_score"] < 0]

        ax.scatter(
            scored_sites["upstream_relative_pos"],
            scored_sites["y"],
            c=scored_sites["ML_score"],
            cmap=cmap,
            norm=norm,
            marker="|",
            s=8,
            linewidths=0.5,
            rasterized=True,
            zorder=2,
        )

        if not unknown_sites.empty:
            ax.scatter(
                unknown_sites["upstream_relative_pos"],
                unknown_sites["y"],
                color="#555555",
                marker="x",
                s=8,
                linewidths=0.5,
                rasterized=True,
                label="ML unavailable",
                zorder=3,
            )
            ax.legend(loc="lower right")

    color_mapper = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    color_mapper.set_array([])

    cbar = fig.colorbar(
        color_mapper,
        ax=ax,
        pad=0.02,
        fraction=0.035,
    )
    cbar.set_label("5mC ML score (modification probability encoding)")
    cbar.set_ticks([0, 64, 128, 192, 255])

    ax.set_xlim(
        -upstream_bp,
        0,
    )
    ax.set_ylim(0, n_reads)

    ax.set_xlabel(
        "Upstream distance from repeat (bp, gene 5′ → 3′; repeat starts at 0)",
        fontsize=12,
        fontweight="bold",
    )
    ax.set_ylabel(
        f"HiFi-overlapping reads sorted by repeat length (n={n_reads:,})",
        fontsize=12,
        fontweight="bold",
    )
    ax.set_title(
        f"{sample_name} {target_locus}: Upstream 2 kb — Single-Molecule 5mC\n"
        "Both modification strands; no ML threshold; gray = covered sequence",
        fontsize=13,
        fontweight="bold",
        loc="left",
    )

    ax.grid(
        axis="x",
        linestyle="--",
        linewidth=0.5,
        alpha=0.2,
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()

    fig.savefig(
        f"{output_prefix}_5mC_tracks.png",
        dpi=300,
        bbox_inches="tight",
    )
    fig.savefig(
        f"{output_prefix}_5mC_tracks.svg",
        bbox_inches="tight",
    )

    plt.show()
    return {"qc": summary_df, "sites": meth_df, "reads": order_df, "figure": fig}

# Independent namespace: earlier notebook variables are left untouched.
als_upstream_2kb = run_als_upstream_2kb()


In [ ]:
# =============================================================================
# PB8–PB11 C9orf72 — Cross-Sample Repeat Composition and Length Comparison
# =============================================================================

# Compare C9orf72 repeat characteristics across four PacBio samples:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# Uses the existing, gene-oriented all_reads_df.
sample_order = ["PB8", "PB9", "PB10", "PB11"] # PB8 is the control, PB9, PB10, and PB11 are the experimental groups 

colors = {
    "PB8": "#404040",
    "PB9": "#595982",
    "PB10": "#F28E2B",
    "PB11": "#17BECF",
}

labels = {
    "PB8": "PB8 (Control)",
    "PB9": "PB9",
    "PB10": "PB10",
    "PB11": "PB11",
}

# Work on a copy; leave all_reads_df unchanged.
plot_df = all_reads_df.loc[
    all_reads_df["sample"].isin(sample_order)
].copy()

valid_length = (
    np.isfinite(plot_df["repeat_length"])
    & (plot_df["repeat_length"] > 0)
)
plot_df = plot_df.loc[valid_length].copy()

for sample in sample_order:
    if not (plot_df["sample"] == sample).any():
        raise ValueError(f"No valid repeat lengths for {sample}")

at_df = plot_df.loc[
    np.isfinite(plot_df["at_percentage"])
].copy()

if at_df.empty:
    raise ValueError("No valid A+T percentages available.")

# Shared limits for the overlay and all four scatter panels.
x_min = at_df["at_percentage"].min()
x_max = at_df["at_percentage"].max()
x_padding = max(0.2, 0.04 * (x_max - x_min))
at_xlim = (x_min - x_padding, x_max + x_padding)

length_min = at_df["repeat_length"].min()
length_max = at_df["repeat_length"].max()
length_ylim = (length_min / 1.25, length_max * 1.25)


def format_at_axis(ax):
    ax.set_yscale("log", base=10)
    ax.set_xlim(at_xlim)
    ax.set_ylim(length_ylim)
    ax.set_xlabel("A+T content in trimmed repeat (%)")
    ax.set_ylabel("Repeat length (bp; log scale)")
    ax.set_axisbelow(True)
    ax.grid(which="major", linestyle="--", alpha=0.3)
    ax.grid(which="minor", axis="y", linestyle=":", alpha=0.15)


# ------------------------------------------------------------
# 1. Overlay: all four samples
# ------------------------------------------------------------

fig_overlay, ax = plt.subplots(figsize=(9, 6.5))

# Shuffle so no sample is consistently drawn on top.
shuffled = at_df.sample(frac=1, random_state=42)

ax.scatter(
    shuffled["at_percentage"],
    shuffled["repeat_length"],
    c=shuffled["sample"].map(colors),
    s=12,
    alpha=0.3,
    edgecolors="none",
    rasterized=True,
)

for sample in sample_order:
    n = int((at_df["sample"] == sample).sum())
    ax.scatter(
        [], [],
        color=colors[sample],
        s=30,
        label=f"{labels[sample]}; n={n:,}",
    )

format_at_axis(ax)
ax.set_title(
    "C9orf72: A+T content versus repeat length\n"
    "PB8–PB11 HiFi-overlapping reads"
)
ax.legend(fontsize=9)

fig_overlay.tight_layout()
plt.show()


# ------------------------------------------------------------
# 2. Four sample-specific panels with identical axis limits
# ------------------------------------------------------------

fig_individual, axes = plt.subplots(
    2, 2,
    figsize=(13, 10),
    sharex=True,
    sharey=True,
)

for ax, sample in zip(axes.flat, sample_order):
    sample_at = at_df.loc[at_df["sample"] == sample]

    ax.scatter(
        sample_at["at_percentage"],
        sample_at["repeat_length"],
        color=colors[sample],
        s=12,
        alpha=0.3,
        edgecolors="none",
        rasterized=True,
    )

    format_at_axis(ax)
    ax.set_title(
        f"{labels[sample]} — n={len(sample_at):,}",
        color=colors[sample],
        fontweight="bold",
    )
    ax.tick_params(
        axis="both", which="both",
        labelbottom=True, labelleft=True,
    )

fig_individual.suptitle(
    "C9orf72: A+T content versus repeat length by sample\n"
    "HiFi-overlapping reads; shared axis limits",
    fontsize=13,
)

fig_individual.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()


# ------------------------------------------------------------
# 3. Overlay: cumulative repeat-length distribution
# ------------------------------------------------------------

fig_cumulative, ax = plt.subplots(figsize=(9, 6.5))

x_end = plot_df["repeat_length"].max() * 1.05

for sample in sample_order:
    lengths = np.sort(
        plot_df.loc[
            plot_df["sample"] == sample, "repeat_length"
        ].to_numpy()
    )

    unique_lengths, counts = np.unique(
        lengths, return_counts=True
    )
    cumulative_percent = 100 * np.cumsum(counts) / len(lengths)

    ax.step(
        np.r_[0, unique_lengths, x_end],
        np.r_[0, cumulative_percent, 100],
        where="post",
        color=colors[sample],
        linewidth=2,
        label=f"{labels[sample]}; n={len(lengths):,}",
    )

ax.set(
    xlabel="Trimmed repeat-length threshold (bp)",
    ylabel="Reads with repeat length ≤ threshold (%)",
    title="C9orf72: cumulative repeat-length distribution\n"
          "PB8–PB11 HiFi-overlapping reads",
    xlim=(0, x_end),
    ylim=(0, 101),
)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
ax.set_yticks(np.arange(0, 101, 10))
ax.set_axisbelow(True)
ax.grid(linestyle="--", alpha=0.3)
ax.legend(loc="lower right", fontsize=9)

fig_cumulative.tight_layout()
plt.show()


# Three complementary visualizations are generated:
#
# FIGURE 1
#     Overlay of A+T content versus repeat length for all four samples.
#
# FIGURE 2
#     Four sample-specific A+T-versus-length panels using identical
#     x- and y-axis limits.
#
# FIGURE 3
#     Cumulative distribution of repeat length for all four samples.
#
# Together, these figures address two related questions:
#
#     1. Does A+T content change with repeat length, and does this pattern
#        differ visually among samples?
#
#     2. How do the overall repeat-length distributions differ among
#        PB8, PB9, PB10, and PB11?

In [ ]:
# A/T Distribution plot: 
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import pysam

spanning_bam = Path(
    "*.trgt.sorted.spanning.bam"
)
hifi_bam = Path("PB*.hifi.pbmm2.bam")

target_locus = "FTDALS1_C9orf72"
target_strand = "-"
max_display_reads = 4000
fl_size = 50

trantab = str.maketrans("ACGTNacgtn", "TGCANtgcan")

def revcomp(seq):
    return seq.translate(trantab)[::-1]


# 1. Collect C9orf72 spanning sequences.
target_sequences = {}

with pysam.AlignmentFile(spanning_bam, "rb", check_sq=False) as bam:
    for read in bam.fetch(until_eof=True):
        if not read.has_tag("TR"):
            continue
        if read.get_tag("TR") != target_locus:
            continue
        if read.is_secondary or read.is_supplementary:
            continue
        if not read.query_sequence:
            continue

        target_sequences[read.query_name] = read.query_sequence.upper()

if not target_sequences:
    raise ValueError("No C9orf72 spanning sequences found.")

# 2. Find exact read-ID overlap with the HiFi BAM.
# No base-quality, read-quality, or mapping-quality threshold.
hifi_overlap_ids = set()

with pysam.AlignmentFile(hifi_bam, "rb", check_sq=False) as bam:
    for read in bam.fetch(until_eof=True):
        if read.query_name in target_sequences:
            hifi_overlap_ids.add(read.query_name)

print(f"C9orf72 spanning reads: {len(target_sequences):,}")
print(f"Also present in HiFi BAM: {len(hifi_overlap_ids):,}")
print(
    "Absent from HiFi BAM: "
    f"{len(set(target_sequences) - hifi_overlap_ids):,}"
)


# 3. Use the spanning sequence to define the repeat region.
# HiFi membership determines which reads are retained.
overlap_tracks = []
short_reads = 0

for read_id in sorted(hifi_overlap_ids):
    seq = target_sequences[read_id]

    # Same gene-orientation convention as your original template.
    if target_strand == "-":
        seq = revcomp(seq)

    # Remove exactly 50 bp from each end.
    if len(seq) <= 2 * fl_size:
        short_reads += 1
        continue

    core_seq = seq[fl_size:-fl_size]

    overlap_tracks.append({
        "read_id": read_id,
        "length": len(core_seq),
        "pos_A": [i for i, base in enumerate(core_seq) if base == "A"],
        "pos_T": [i for i, base in enumerate(core_seq) if base == "T"],
    })

if not overlap_tracks:
    raise ValueError("No overlapping reads remain after flank removal.")

df_hifi_tracks = (
    pd.DataFrame(overlap_tracks)
    .sort_values(["length", "read_id"])
    .reset_index(drop=True)
)

if len(df_hifi_tracks) > max_display_reads:
    idx = np.linspace(
        0, len(df_hifi_tracks) - 1,
        max_display_reads, dtype=int
    )
    df_hifi_plot = df_hifi_tracks.iloc[idx].reset_index(drop=True)
else:
    df_hifi_plot = df_hifi_tracks.copy()

print(f"Too short to remove both flanks: {short_reads:,}")
print(f"Reads plotted: {len(df_hifi_plot):,}")


# 4. Plot repeat backbones and A/T positions.
fig, ax = plt.subplots(figsize=(12, 8))

y_positions = np.arange(1, len(df_hifi_plot) + 1)

ax.hlines(
    y=y_positions,
    xmin=0,
    xmax=df_hifi_plot["length"].to_numpy(),
    color="#BDC3C7",
    linewidth=1.0,
    alpha=0.6,
    zorder=1,
)

for base, color in [("A", "#2E8B57"), ("T", "#F39C12")]:
    x_positions = []
    base_y_positions = []

    for y_pos, positions in zip(
        y_positions, df_hifi_plot[f"pos_{base}"]
    ):
        x_positions.extend(positions)
        base_y_positions.extend([y_pos] * len(positions))

    ax.scatter(
        x_positions,
        base_y_positions,
        color=color,
        s=6,
        marker="|",
        linewidths=0.6,
        rasterized=True,
        zorder=3,
    )

ax.set_xlabel(
    "Repeat Coordinate (bp, gene 5′ → 3′)",
    fontsize=11,
    fontweight="bold",
)
ax.set_ylabel(
    f"Spanning Reads (Sorted by Repeat Length, N={len(df_hifi_plot):,})",
    fontsize=11,
    fontweight="bold",
)
ax.set_title(
    f"PB* {target_locus}: A/T Tracks\n"
    "Spanning reads also present in HiFi BAM",
    fontsize=12,
    fontweight="bold",
    loc="left",
)

ax.set_ylim(0, len(df_hifi_plot) + 1)
ax.set_xlim(0, df_hifi_plot["length"].max() * 1.02)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", linestyle="--", alpha=0.3)

legend_elements = [
    Line2D(
        [0], [0],
        marker="|",
        color=color,
        label=f"{base} inside Repeat",
        linestyle="None",
        markersize=10,
        markeredgewidth=1.5,
    )
    for base, color in [("A", "#2E8B57"), ("T", "#F39C12")]
]

ax.legend(
    handles=legend_elements,
    loc="lower right",
    bbox_to_anchor=(0.96, 0.05),
    frameon=True,
    facecolor="white",
    edgecolor="#CCCCCC",
    fontsize=10,
)

plt.tight_layout()
plt.savefig(
    "PB*_C9orf72_HiFi_overlap_AT_tracks.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Checking SNP in ALS samples: (Exploring)

In [ ]:
# Full mapped HiFi BAM SEQ is retained in reference-plus 5′→3′ orientation. 
# No reverse-complement function is used. Soft-clipped bases remain available. 
# Match the complete TRGT segment exactly once, use FL to locate the repeat, and extract full left/right flanks. 
# Hard-clipped and unmapped reads are excluded because full sequence or reference orientation is unavailable. 
# Only the HiFi BAM is used; unmatched TRGT IDs are listed in QC.


In [ ]:
# Coordinate convention: 27573484 and 27573546 are zero-based BED boundaries, with the end exclusive. 
# The first repeat read base is q_start; the last is q_end−1. 
# The right flank starts at q_end. 
# In ordinary one-based hg38 notation, the catalog repeat bases are 27573485–27573546. 
# These fixed reference boundaries are annotation anchors regardless of allele length; they do not change the measured read repeat length.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import numpy as np
import pandas as pd
import pysam
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from IPython.display import display

DATA_DIR = Path('*')
TRGT_BAM = DATA_DIR / '*.bc2032.trgt.sorted.spanning.bam'
HIFI_BAM = DATA_DIR / 'PB*.hifi.pbmm2.bam'
SNV_CSV = Path('C9orf72_Common_SNPs.csv')
OUTPUT_DIR = Path.cwd() / 'PB*_C9orf72_full_flank_results'
CHROM = 'chr9'
REPEAT_START, REPEAT_END = 27573484, 27573546  # defined repeat region from ptcp pipeline 
LOCUS = 'FTDALS1_C9orf72'
FLANK_BP = 3000
FOCUS_RSIDS = None
MIN_MAPQ = 0
MIN_BASEQ = 0
# Optional local hg38 FASTA with .fai; validates every selected REF base.
HG38_FASTA = None
SAVE_FASTA = True

for p in (TRGT_BAM, HIFI_BAM, SNV_CSV):
    if not p.is_file():
        raise FileNotFoundError(p)
assert FLANK_BP > 0 and REPEAT_END > REPEAT_START
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
snvs = pd.read_csv(SNV_CSV)

snvs.columns = snvs.columns.str.lstrip('#').str.strip('"')

snvs = snvs.loc[(snvs['class'] == 'snv') & (snvs['chrom'] == CHROM)].copy() # keep only chr9 SNVs

# Verify that every selected record is exactly one base long
assert ((snvs.chromEnd - snvs.chromStart) == 1).all()

# Verify that all REF alleles are canonical nucleotides
assert snvs.ref.isin(list('ACGT')).all()

# Identify SNVs located INSIDE the repeat interval
inside = (snvs.chromStart >= REPEAT_START) & (snvs.chromStart < REPEAT_END)


print(f'SNVs in CSV on {CHROM}: {len(snvs)}; excluded inside repeat: {inside.sum()}')

# Select SNVs within ±3 kb of the repeat
snvs = snvs.loc[~inside & (snvs.chromStart >= REPEAT_START-FLANK_BP)
                & (snvs.chromStart < REPEAT_END+FLANK_BP)].copy()

# Optionally restrict the analysis to specific rsIDs
if FOCUS_RSIDS is not None:
    missing = set(FOCUS_RSIDS) - set(snvs.name)
    if missing:
        raise ValueError(f'Requested IDs unavailable in selected flanks: {sorted(missing)}')
    snvs = snvs.loc[snvs.name.isin(FOCUS_RSIDS)].copy()

# Sort selected SNVs by genomic position
snvs = snvs.sort_values(['chromStart', 'name']).reset_index(drop=True)

# Confirm that at least one flanking SNV remains
if snvs.empty:
    raise ValueError('No flanking SNVs selected. Increase FLANK_BP or change FOCUS_RSIDS.')

# Require one SNV annotation per reference position
if snvs.chromStart.duplicated().any():
    raise ValueError('Multiple SNV rows at the same reference position; resolve before plotting.')

# Classify SNVs as 5′ or 3′ of the repeat
snvs['side'] = np.where(snvs.chromStart < REPEAT_START, '5prime', '3prime')

# Convert each SNV to a conventional 1-based genomic position
snvs['pos1'] = snvs.chromStart + 1

# Calculate SNV position relative to the repeat boundary
snvs['reference_offset_bp'] = np.where(snvs.side == '5prime',
    snvs.chromStart-REPEAT_START, snvs.chromStart-REPEAT_END)

# Optionally validate REF alleles against hg38
if HG38_FASTA:
    with pysam.FastaFile(str(HG38_FASTA)) as fa:
        observed = [fa.fetch(CHROM, int(s), int(s)+1).upper() for s in snvs.chromStart]
    if observed != snvs.ref.tolist():
        raise ValueError('REF mismatch: check assembly, coordinates, and CSV.')
snvs.to_csv(OUTPUT_DIR / 'selected_flanking_snvs.csv', index=False)
display(snvs[['name','pos1','ref','alts','side','reference_offset_bp']])

In [ ]:
# C9orf72 Flanking-SNV Analysis
# Part 2 — TRGT/HiFi Read Matching, Repeat Anchoring, and Flank Extraction

# Starting from the TRGT C9orf72 spanning reads, identify the corresponding
# primary HiFi records and determine the exact repeat boundaries within
# each original HiFi read.
#
# This stage performs strict read-level QC before downstream SNV extraction.


trgt_records = {}
with pysam.AlignmentFile(str(TRGT_BAM), 'rb') as bam:
    for r in bam.fetch(until_eof=True):
        if r.has_tag('TR') and r.get_tag('TR') == LOCUS:
            if r.query_name in trgt_records:
                raise ValueError(f'Duplicate TRGT locus record: {r.query_name}')
            trgt_records[r.query_name] = r
if not trgt_records:
    raise ValueError(f'No TR={LOCUS} records.')

hifi_records = defaultdict(list)

supplementary_counts = Counter()

with pysam.AlignmentFile(str(HIFI_BAM), 'rb') as bam:
    # Header names/lengths alone cannot distinguish hg19 from hg38 reliably.
    if CHROM not in bam.references:
        raise ValueError(f'{CHROM} absent from HiFi BAM header')
    for r in bam.fetch(until_eof=True):
        if r.query_name not in trgt_records:
            continue
        if r.is_supplementary:
            supplementary_counts[r.query_name] += 1
        elif not r.is_secondary:
            hifi_records[r.query_name].append(r)
qc_rows, eligible, sequences = [], {}, {}

for rid, tr in trgt_records.items():
    qc = dict(read_id=rid, status='ok', supplementary_records=supplementary_counts[rid])
    records = hifi_records.get(rid, [])
    if len(records) != 1:
        qc['status'] = 'not_in_hifi' if not records else 'duplicate_primary'
    else:
        r = records[0]
        qc.update(alignment_strand='reverse' if r.is_reverse else 'forward',
                  mapq=r.mapping_quality, full_read_length=r.query_length)
        if r.is_unmapped or r.reference_name != CHROM:
            qc['status'] = 'unmapped_or_other_contig'
        elif not r.query_sequence or not tr.query_sequence:
            qc['status'] = 'missing_sequence'
        elif any(op == 5 for op, length in (r.cigartuples or [])):
            qc['status'] = 'hard_clipped'
        elif tr.is_unmapped or tr.reference_name != CHROM:
            qc['status'] = 'trgt_orientation_unknown'
        elif not tr.has_tag('FL') or len(tr.get_tag('FL')) != 2:
            qc['status'] = 'missing_or_invalid_FL'
        else:
            seq, segment = r.query_sequence.upper(), tr.query_sequence.upper()
            fl, fr = map(int, tr.get_tag('FL'))
            if min(fl, fr) < 0 or fl+fr >= len(segment):
                qc['status'] = 'invalid_FL'
            else:
                p = seq.find(segment)
                if p < 0:
                    qc['status'] = 'trgt_segment_not_found_same_orientation'
                elif seq.find(segment, p+1) >= 0:
                    qc['status'] = 'ambiguous_segment_match'
                else:
                    qs, qe = p+fl, p+len(segment)-fr
                    qc.update(repeat_start_q0=qs, repeat_end_q0=qe,
                              repeat_length_bp=qe-qs,
                              trgt_allele=tr.get_tag('AL') if tr.has_tag('AL') else None)
                    assert seq[qs:qe] == segment[fl:len(segment)-fr]
                    eligible[rid] = r
                    sequences[rid] = seq
    qc_rows.append(qc)
    
read_qc = pd.DataFrame(qc_rows)
read_qc.to_csv(OUTPUT_DIR / 'read_qc.csv', index=False)
reads = read_qc.loc[read_qc.status == 'ok'].sort_values(
    ['repeat_length_bp','read_id']).reset_index(drop=True)

if reads.empty:
    raise ValueError('No eligible reads. Inspect read_qc.csv.')
print(f'TRGT IDs: {len(trgt_records):,}; eligible HiFi reads: {len(reads):,}')
display(read_qc.status.value_counts().rename('reads').to_frame())
display(reads.groupby('alignment_strand').repeat_length_bp.agg(['count','min','median','max']))

if SAVE_FASTA:
    with (OUTPUT_DIR / 'hifi_reads_reference_plus.fasta').open('w') as f:
        for row in reads.itertuples():
            seq = sequences[row.read_id]
            f.write(f'>{row.read_id} repeat_bp={int(row.repeat_length_bp)} alignment={row.alignment_strand}\n')
            for i in range(0,len(seq),80):
                f.write(seq[i:i+80]+'\n')

# Extract every available flank base, including soft-clipped sequence.
# No reverse-complement operation is applied anywhere in this notebook.
reads['left_available_bp'] = reads.repeat_start_q0.astype(int)
reads['right_available_bp'] = reads.full_read_length.astype(int)-reads.repeat_end_q0.astype(int)
reads['left_plot_bp'] = np.minimum(reads.left_available_bp, FLANK_BP)
reads['right_plot_bp'] = np.minimum(reads.right_available_bp, FLANK_BP)
reads['left_missing_bp'] = FLANK_BP-reads.left_plot_bp
reads['right_missing_bp'] = FLANK_BP-reads.right_plot_bp
reads.to_csv(OUTPUT_DIR/'flank_qc.csv',index=False)
for side in ['left','right']:
    with (OUTPUT_DIR/f'{side}_full_flanks_reference_plus.fasta').open('w') as f:
        for x in reads.itertuples():
            seq=sequences[x.read_id]
            flank=seq[:int(x.repeat_start_q0)] if side=='left' else seq[int(x.repeat_end_q0):]
            f.write(f'>{x.read_id} flank_bp={len(flank)} repeat_bp={int(x.repeat_length_bp)}\n')
            for j in range(0,len(flank),80): f.write(flank[j:j+80]+'\n')
display(reads[['read_id','repeat_length_bp','left_available_bp','right_available_bp']].head())


In [ ]:
# C9orf72 Flanking-SNV Analysis
# Part 3 — Direct Repeat-Boundary-Based SNV Sampling

# PURPOSE
# -------
# Sample the nucleotide carried by each individual HiFi molecule at the
# expected position of each catalog flanking SNV.
#
# Instead of using the primary BAM alignment to locate each SNV, this
# analysis uses the C9orf72 repeat itself as the positional anchor.

def sample_offset(seq, repeat_start_q0, repeat_end_q0, side, offset):
    """Apply a catalog offset directly to a physical read boundary; no alignment."""
    if side=='5prime':
        q=repeat_start_q0+offset
        valid=0<=q<repeat_start_q0
    else:
        q=repeat_end_q0+offset
        valid=repeat_end_q0<=q<len(seq)
    return (q,seq[q]) if valid else (None,None)

# Boundary convention tests: last left base=-1; first right base=0.
assert sample_offset('ACGGGT',2,5,'5prime',-1)==(1,'C')
assert sample_offset('ACGGGT',2,5,'3prime',0)==(5,'T')
assert sample_offset('ACGGGT',2,5,'5prime',-3)==(None,None)
assert sample_offset('ACGGGT',2,5,'3prime',1)==(None,None)

call_rows=[]
neighbor_rows=[]
for row in reads.itertuples():
    r=eligible[row.read_id]
    seq=sequences[row.read_id]
    qs,qe=int(row.repeat_start_q0),int(row.repeat_end_q0)
    
    # Existing primary mapping is used ONLY as a diagnostic comparator.
    primary_map={p:q for q,p in r.get_aligned_pairs() if p is not None}
    
    for snv in snvs.itertuples():
        offset=int(snv.reference_offset_bp)
        q,base=sample_offset(seq,qs,qe,snv.side,offset)
        quality=int(r.query_qualities[q]) if q is not None and r.query_qualities is not None else None
        status=('uncovered' if q is None else
                'low_quality' if quality is None or quality<MIN_BASEQ or r.mapping_quality<MIN_MAPQ else
                'ambiguous_base' if base not in 'ACGT' else 'called')
        
        primary_q=primary_map.get(int(snv.chromStart))
        
        allowed={snv.ref}|set(str(snv.alts).rstrip(',').split(','))
        
        call_rows.append(dict(read_id=row.read_id,alignment_strand=row.alignment_strand,
            repeat_length_bp=int(row.repeat_length_bp),snv=snv.name,side=snv.side,
            chrom=CHROM,pos0=int(snv.chromStart),pos1=int(snv.pos1),ref=snv.ref,alts=snv.alts,
            reference_offset_bp=offset,read_offset_bp=offset,q0=q,q1=None if q is None else q+1,
            observed_base=base,baseq=quality,mapq=r.mapping_quality,status=status,
            in_catalog_alleles=None if base is None else base in allowed,
            method='direct_read_boundary_offset',primary_aligned_q0=primary_q,
            direct_minus_primary_q=None if q is None or primary_q is None else q-primary_q))
        
        for delta in [-1,0,1]:
            nq,nb=sample_offset(seq,qs,qe,snv.side,offset+delta)
            neighbor_rows.append(dict(read_id=row.read_id,snv=snv.name,side=snv.side,
                repeat_length_bp=int(row.repeat_length_bp),delta=delta,
                sampled_read_offset_bp=offset+delta,q0=nq,observed_base=nb,
                baseq=None if nq is None or r.query_qualities is None else int(r.query_qualities[nq])))

calls=pd.DataFrame(call_rows)
calls.to_csv(OUTPUT_DIR/'per_read_snv_offset_samples.csv',index=False)

neighbors=pd.DataFrame(neighbor_rows)
neighbors.to_csv(OUTPUT_DIR/'snv_offset_plus_minus_one.csv',index=False)

summary=calls.groupby(['snv','side','pos1','ref','status','observed_base'],dropna=False).size().rename('n_reads').reset_index()
summary.to_csv(OUTPUT_DIR/'snv_summary.csv',index=False)

display(calls.status.value_counts().rename('observations').to_frame())

print('Primary alignment comparison: zero means agreement; nonzero means a positional difference.')
display(calls.direct_minus_primary_q.value_counts(dropna=False).rename('observations').to_frame().head(15))
display(summary.head(20))

In [ ]:
# Generating the Common SNV figures 

BASE_COLORS = {'A':'#2CA25F', 'T':'#E34A33', 'C':'#3182BD', 'G':'#E6AB02'}
STATES = list(BASE_COLORS) + ['uncovered','low_quality','deletion','ref_skip','ambiguous_base','boundary_conflict']
COLORS = list(BASE_COLORS.values()) + ['#F2F2F2','#BDBDBD','#333333','#9163B6','#DDA0DD','#00BFC4']
cmap = ListedColormap(COLORS)
norm = BoundaryNorm(np.arange(-.5,len(STATES)+.5),len(STATES))
state_code = {state:i for i,state in enumerate(STATES)}

def encode_call(x):
    return state_code[x.observed_base if x.status == 'called' else x.status]
calls['color_code'] = [encode_call(x) for x in calls.itertuples()]
matrix = calls.pivot(index='read_id',columns='snv',values='color_code').reindex(
    index=reads.read_id,columns=snvs.name).to_numpy(dtype=int)

def plot_snvs(selected_reads, mode='heatmap', title_suffix='all alignments'):
    if selected_reads.empty:
        return None
    indices = reads.index[reads.read_id.isin(selected_reads.read_id)].to_numpy()
    data = matrix[indices]
    rr = reads.loc[indices].reset_index(drop=True)
    n = len(rr)
    fig, axes = plt.subplots(1,4,figsize=(19,9),sharey=True,
        gridspec_kw={'width_ratios':[5,.85,5,1.6],'wspace':.08})
    left, middle, right, length_ax = axes
    for side, ax, boundary in [('5prime',left,REPEAT_START),('3prime',right,REPEAT_END)]:
        cols = np.flatnonzero(snvs.side.to_numpy() == side)
        subset = snvs.iloc[cols]
        if len(cols):
            if mode == 'heatmap':
                ax.imshow(data[:,cols],cmap=cmap,norm=norm,aspect='auto',
                          interpolation='nearest',origin='upper',extent=[-.5,len(cols)-.5,n-.5,-.5],rasterized=True)
                ax.scatter(np.arange(len(cols)),np.full(len(cols),-2),
                    c=[BASE_COLORS[b] for b in subset.ref],marker='s',s=45,clip_on=False)
                ax.set_xticks(np.arange(len(cols)))
                ax.set_xticklabels([f'{x.name}\n{x.reference_offset_bp:+d} bp' for x in subset.itertuples()],
                                   rotation=90,fontsize=8)
                ax.set_xlabel('SNV sites (equal spacing); labels give hg38 offsets')
            else:
                x = subset.reference_offset_bp.to_numpy()
                xx = np.tile(x,n); yy = np.repeat(np.arange(n),len(cols))
                codes = data[:,cols].ravel()
                # White background represents uncovered observations.
                keep = codes != state_code['uncovered']
                ax.scatter(xx[keep],yy[keep],c=codes[keep],cmap=cmap,norm=norm,
                           marker='s',s=5,linewidths=0,rasterized=True)
                ax.scatter(x,np.full(len(x),-2),c=[BASE_COLORS[b] for b in subset.ref],
                           marker='s',s=35,clip_on=False)
                ax.set_xlim((-FLANK_BP,0) if side == '5prime' else (0,FLANK_BP))
                ax.set_xlabel('Reference offset from boundary (bp)')
        else:
            ax.text(.5,.5,'No selected SNVs',transform=ax.transAxes,ha='center')
        ax.set_title(f"Plus-strand {'5′ left' if side == '5prime' else '3′ right'} flank\nBED boundary {boundary:,}",pad=28)
        ax.set_facecolor('#F2F2F2')
    middle.set_facecolor('#E1E1E1')
    middle.text(.5,.5,'Repeat\ncompressed\n\n27573484\nto\n27573546\n(BED)',
                transform=middle.transAxes,ha='center',va='center',fontsize=9)
    middle.set_xticks([])
    middle.set_title('C9orf72',pad=28)
    length_ax.plot(rr.repeat_length_bp,np.arange(n),color='#555555',linewidth=1)
    length_ax.set_xscale('log')
    length_ax.set_xlabel('Repeat bp\n(log scale)')
    length_ax.set_title('TRGT length',pad=28)
    ticks = np.unique(np.linspace(0,n-1,min(n,9)).astype(int))
    left.set_yticks(ticks)
    left.set_yticklabels([f'{t+1:,} | {int(rr.iloc[t].repeat_length_bp):,} bp' for t in ticks])
    left.set_ylabel('Read rank | repeat length (shortest → longest)')
    for ax in axes:
        ax.set_ylim(n-.5,-.5)
    for ax in axes[1:]:
        ax.tick_params(axis='y',left=False,labelleft=False)
    fig.suptitle(f'PB* C9orf72 SNV-offset samples — {title_suffix} — N={n:,}\nDirect read-boundary offsets (no flank alignment); plus 5′→3′; top squares = CSV REF',y=.99)
    fig.legend([Patch(facecolor=c) for c in COLORS],STATES,loc='upper center',
               bbox_to_anchor=(.5,.93),ncol=5,frameon=False,fontsize=9)
    fig.subplots_adjust(top=.77,bottom=.31 if mode=='heatmap' else .12,left=.12,right=.98)
    stem = OUTPUT_DIR / f"PB*_C9orf72_{mode}_{title_suffix.replace(' ','_')}"
    fig.savefig(str(stem)+'.png',dpi=220,bbox_inches='tight')
    fig.savefig(str(stem)+'.pdf',bbox_inches='tight')
    plt.show()
    return fig

plot_snvs(reads,'heatmap')
plot_snvs(reads,'distance')
# Strand groups retain the same plus-oriented alleles and ascending repeat-length order.
for strand in ['forward','reverse']:
    plot_snvs(reads.loc[reads.alignment_strand == strand], 'heatmap', strand+' alignments')